<div dir="rtl">

# 🇮🇶 دفتر التدريب الرئيسي — Seed-VC العراقي (العدة القصوى)

**أقوى وأضمن نسخة لتدريب نموذج تحويل الصوت على اللهجة العراقية على Colab المجاني.**

هذا الدفتر مستقل ذاتياً: كل أدواته (المدرب المحصّن، الحارس، المراقب، المقيّم)
مضمّنة نصياً في خلاياه — لا تحتاج رفع أي ملف، وتعمل من أول تشغيل.

## ماذا يعالج هذا الدفتر (مقابل train.py الرسمي)؟
| المشكلة في الرسمي | الحل هنا |
|---|---|
| الاستئناف يصفّر عدّاد الخطوات (يحمّل الأوزان فقط) → نقاط الجلسة الجديدة تُحذف صامتةً | **استئناف كامل** (أوزان + مُحسِّن + مجدول + خطوات) — الترقيم متصل عبر الجلسات |
| الحفظ غير ذرّي — انقطاع أثناء torch.save يفسد النقطة | حفظ **ذرّي** (tmp → rename) + مرآة Drive فورية |
| طباعات min/max تُغرق السجل | إسكات تلقائي (سطران فقط في modules/audio.py) |
| لا حماية من NaN | **حارس NaN**: تخطّي الدفعة، وثلاث متتاليات = توقف نظيف |
| لا يعرف متى تنتهي الجلسة | **ميزانية زمنية**: يوقف نفسه ويحفظ قبل نفاد الجلسة |
| لا مقياس "أي نقطة أفضل" | **تقييم موضوعي**: SIM (CAMPPlus) + UTMOS + خط أساس رسمي |

## خريطة الجلسة (اقرأها مرة واحدة)
1. خلايا 1–2: الإعدادات وDrive — **كل جلسة**
2. خلية 3: البيئة — **كل جلسة جديدة فقط** (بعد انقطاع)
3. خلايا 4–6: العدة والبيانات والعزل — أول مرة فقط، وتتجاوز نفسها عند التكرار
4. خلية 7: **التدريب** — اتركها تعمل؛ الحارس يستأنف تلقائياً
5. خلية 8: المراقبة — شغّلها في أي لحظة بجانب التدريب
6. خلية 9: التقييم — **جلسة مستقلة بعد التدريب** (لا أثناءه)
7. خلية 10: الاعتماد والنشر — بعد اختيار النقطة الذهبية

> ⚠️ **قاعدة ذهبية**: الرقم المطبوع في السجل هو EMA منعّم 0.99 — استقراره
> حول 0.59 طبيعي تماماً. القرار "أي نقطة أفضل" يتخذ بالتقييم الموضوعي (خلية 9) فقط.

</div>

<div dir="rtl">

## الخلية 1 — الإعدادات المركزية

كل المقابض في مكان واحد وتُحفظ على Drive (تُستعاد تلقائياً عند الاستئناف).
</div>

In [ ]:
# ============================================================ #
#  الخلية 1 — الإعدادات المركزية والأدوات المساعدة             #
# ============================================================ #
import os, sys, json, subprocess, shutil, time, re
from pathlib import Path

NB_VERSION = "ultimate-1.0"

def _pick_root():
    # الجذر الدائم: Drive موصّل، وإلا مؤقت محلي
    for cand in ("/content/drive/MyDrive/iraqi_seedvc",):
        p = Path(cand)
        if p.exists():
            return p
    override = os.environ.get("IRAQI_DATA_ROOT", "").strip()
    if override:
        return Path(override)
    for cand in ("/workspace", "/root", "/content"):
        p = Path(cand)
        try:
            (p / ".write_test").write_text("ok"); (p / ".write_test").unlink()
            return p / "iraqi_seedvc"
        except Exception:
            continue
    return Path.home() / "iraqi_seedvc"

ROOT = _pick_root()
ROOT.mkdir(parents=True, exist_ok=True)
CFG_PATH = ROOT / "nb_config.json"

DEFAULTS = dict(
    # ---- التدريب ----
    RUN_NAME        = "iraqi_v1",
    TOTAL_STEPS     = 6000,   # الهدف التراكمي (3000 بوابة أولى، 6000، ثم 10000 إن تحسّن)
    BATCH           = 2,      # الرسمي يوصي 2
    SAVE_EVERY      = 500,    # نقطة كل ~19 دقيقة على T4
    KEEP_CKPTS      = 2,      # محلياً وعلى Drive (سعة 15GB)
    TRAIN_WORKERS   = 2,
    TIME_BUDGET_H   = 11,     # ساعات الجلسة قبل التوقف النظيف (سقف كولاب 12)
    LR              = 1e-5,   # مثبت الرسمي — لا تغيّره
    # ---- البيانات ----
    DATA_DIR_NAME   = "data",        # مجلد البيانات داخل ROOT
    HOLDOUT_COUNT   = 24,            # مقاطع اختبار معزولة
    # ---- التقييم ----
    EVAL_MAX_CKPTS  = 6,
    EVAL_SOURCES    = 3,
    EVAL_REFS       = 2,
    DIFF_STEPS      = 30,            # خطوات الانتشار للتقييم — ثبّتها للمقارنة العادلة
    # ---- البنية ----
    SEEDVC_DIR      = "seed-vc",     # داخل ROOT
    KIT_DIR_NAME    = "kit",         # مجلد العدة داخل ROOT
)

# استعادة/دمج الإعدادات المحفوظة
CFG = dict(DEFAULTS)
if CFG_PATH.exists():
    try:
        saved = json.loads(CFG_PATH.read_text(encoding="utf-8"))
        CFG.update({k: v for k, v in saved.items() if k in DEFAULTS})
        print(f"[cfg] استُعيدت الإعدادات المحفوظة (نسخة {saved.get('_v','?')})")
    except Exception as e:
        print(f"[cfg] تعذر قراءة الإعدادات المحفوظة: {e} — استخدام الافتراضي")
CFG["_v"] = NB_VERSION

SEEDVC = ROOT / CFG["SEEDVC_DIR"]
KIT_DIR = ROOT / CFG["KIT_DIR_NAME"]
DATA_DIR = ROOT / CFG["DATA_DIR_NAME"]
RUN_DIR  = ROOT / "runs" / CFG["RUN_NAME"]
TEST_DIR = ROOT / "holdout_test"

CFG_PATH.write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding="utf-8")

def sh(cmd, **kw):
    """تشغيل أمر مع طباعة نظيفة."""
    print("$", " ".join(str(c) for c in cmd) if isinstance(cmd, (list, tuple)) else cmd)
    r = subprocess.run(cmd, shell=isinstance(cmd, str), **kw)
    if r.returncode != 0:
        print(f"[خطأ] رمز الخروج {r.returncode}")
    return r

print(f"[cfg] الجذر: {ROOT}")
print(f"[cfg] المستودع: {SEEDVC} | العدة: {KIT_DIR}")
print(f"[cfg] البيانات: {DATA_DIR} | التشغيل: {RUN_DIR}")
print(f"[cfg] الهدف التراكمي: {CFG['TOTAL_STEPS']} خطوة | ميزانية الجلسة: {CFG['TIME_BUDGET_H']} ساعة")
if not ROOT.as_posix().startswith("/content/drive"):
    print("[cfg] ⚠ Drive غير موصول — الإعدادات محلية مؤقتة (شغّل الخلية 2 أولاً)")

<div dir="rtl">

## الخلية 2 — توصيل Google Drive (التخزين الدائم)

بدون Drive: كل شيء يفنى عند انقطاع الجلسة. مع Drive: النقاط والسجلات والإعدادات تبقى.
</div>

In [ ]:
# ============================================================ #
#  الخلية 2 — توصيل Drive وتثبيت الجذر الدائم                 #
# ============================================================ #
import os, shutil, json
from pathlib import Path

IN_COLAB = Path("/content").exists()
DRIVE_OK = False

if IN_COLAB:
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
    DRIVE_OK = Path("/content/drive/MyDrive").exists()
else:
    print("لسنا في Colab — نبحث عن مجلد drive محلي بديل")
    for cand in ("/workspace/drive", "./drive"):
        if Path(cand).exists():
            DRIVE_OK = True
            print(f"مجلد بديل: {cand}")

if DRIVE_OK and IN_COLAB:
    NEW_ROOT = Path("/content/drive/MyDrive/iraqi_seedvc")
    NEW_ROOT.mkdir(parents=True, exist_ok=True)
    # ترحيل الإعدادات من الجذر المؤقت إن وُجدت
    old_cfg = Path(os.environ.get("IRAQI_TMP_CFG", "")) if os.environ.get("IRAQI_TMP_CFG") else None
    print(f"[drive] ✓ موصول — الجذر الدائم: {NEW_ROOT}")

# مسارات Drive الثابتة (تُستخدم في كل الخلايا لاحقاً)
DRIVE_ROOT = Path("/content/drive/MyDrive/iraqi_seedvc") if DRIVE_OK and IN_COLAB else None
DRIVE_RUN  = DRIVE_ROOT / "runs" / CFG["RUN_NAME"] if DRIVE_ROOT else None
if DRIVE_RUN:
    DRIVE_RUN.mkdir(parents=True, exist_ok=True)
    print(f"[drive] مجلد التشغيل على Drive: {DRIVE_RUN}")
    # استعادة الإعدادات من Drive إن كانت الجلسة جديدة
    dcfg = DRIVE_ROOT / "nb_config.json"
    if dcfg.exists() and not CFG_PATH.as_posix().startswith("/content/drive"):
        CFG.update(json.loads(dcfg.read_text(encoding="utf-8")))
        CFG_PATH.write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding="utf-8")
        print("[drive] استُعيدت الإعدادات من Drive")
print(f"[drive] DRIVE_OK = {DRIVE_OK}")

<div dir="rtl">

## الخلية 3 — تجهيز البيئة (كل جلسة جديدة بعد انقطاع)

استنساخ seed-vc + تثبيت المتطلبات. **بطاقة T4**: تستغرق 5–10 دقائق أول مرة،
وأقل من دقيقة عند وجود كاش (الخلية تكتشف الموجود وتتجاوزه).
</div>

In [ ]:
# ============================================================ #
#  الخلية 3 — البيئة: seed-vc + الاعتماديات                   #
# ============================================================ #
import subprocess, sys, os, time
from pathlib import Path

t0 = time.time()
SEEDVC.mkdir(parents=True, exist_ok=True)

# ---- 1) استنساخ المستودع (مرة واحدة) ----
if not (SEEDVC / "train.py").exists():
    print("[env] استنساخ seed-vc...")
    r = subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Plachtaa/seed-vc.git", str(SEEDVC)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr); raise RuntimeError("فشل الاستنساخ")
else:
    print(f"[env] المستودع موجود: {SEEDVC}")

# ---- 2) البيئة الافتراضية والمتطلبات ----
VENV = ROOT / "venv"
VENV_PY = VENV / "bin" / "python"
if not VENV_PY.exists():
    print("[env] إنشاء venv وتثبيت المتطلبات (5–10 دقائق أول مرة)...")
    sh(["uv", "venv", str(VENV), "--python", "3.10"])
    req = SEEDVC / "requirements.txt"
    sh([str(VENV_PY), "-m", "pip", "install", "-q", "-r", str(req)])
    # torch بكودا إذا غاب
    sh([str(VENV_PY), "-m", "pip", "install", "-q",
        "torch==2.4.0", "torchaudio==2.4.0",
        "--index-url", "https://download.pytorch.org/whl/cu121"])
    sh([str(VENV_PY), "-m", "pip", "install", "-q", "soundfile", "tqdm"])
else:
    print(f"[env] البيئة موجودة: {VENV_PY}")

# ---- 3) فحص GPU ----
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                    "--format=csv,noheader"], capture_output=True, text=True)
print(f"[env] GPU: {r.stdout.strip() or 'غير متاحة!'}")
assert "T4" in r.stdout or "Tesla" in r.stdout or "A" in r.stdout or "L4" in r.stdout, \
       "لا GPU — غيّر وقت التشغيل إلى GPU"

print(f"[env] جاهز خلال {time.time()-t0:.0f} ثانية")

<div dir="rtl">

## الخلية 4 — تثبيت عدة التدريب المحصّنة (مضمّنة نصياً)

كل ملفات العدة الخمسة مكتوبة هنا داخل الدفتر إلى `ROOT/kit/` — لا تنزيل خارجي.
هذه هي النسخة المحصّنة المطابقة للخطة: استئناف كامل، حفظ ذرّي، حارس NaN، ميزانية زمنية.
</div>

In [ ]:
# ============================================================ #
#  الخلية 4 — كتابة ملفات العدة الستة إلى ROOT/kit            #
#  (مضمّنة بصيغة base64 — محصّنة ضد أي محتوى)                  #
#  تتجاوز الموجود السليم تلقائياً — آمنة لإعادة التشغيل        #
# ============================================================ #
import base64, hashlib, ast
from pathlib import Path

KIT_DIR.mkdir(parents=True, exist_ok=True)

# بصمة كل ملف (sha256[:12]) → المحتوى بـ base64
KIT_FILES = {
    "ckpt_utils.py": "IyAtKi0gY29kaW5nOiB1dGYtOCAtKi0KIiIiCmNrcHRfdXRpbHMucHkg4oCUINil2K/Yp9ix2Kkg2YbZgtin2Lcg2KfZhNiq2K3ZgtmCOiDYrdmB2Lgg2LDYsdmR2YogKyDZhdix2KLYqSBEcml2ZSArINiq2YLZhNmK2YUg2KLZhdmGCj09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CtmF2LXZhdmFINmE2YrYudmF2YQg2YXYuSDZhdiz2KrZiNiv2Lkgc2VlZC12YyDYp9mE2LHYs9mF2Yog2K/ZiNmGINij2Yog2KrYudiv2YrZhCDZgdmK2YcuCtmD2YQg2KfZhNiv2YjYp9mEINmF2LPYqtmC2YTYqSAo2YTYpyDYqti52KrZhdivINi52YTZiSDZhdiq2LrZitix2KfYqiDZhtmI2KrYqNmI2YMg2K7Yp9ix2KzZitipKS4KCtin2YTYttmF2KfZhtin2Ko6CiAg4oCiINit2YHYuCDYsNix2ZHZiiAodG1wIOKGkiByZW5hbWUpOiDZhNinINmG2YLYt9ipINmG2LXZgSDZhdmD2KrZiNio2Kkg2K3YqtmJINmE2Ygg2KfZhtmC2LfYudiqINin2YTYrNmE2LPYqSDYo9ir2YbYp9ihINin2YTYrdmB2LgKICDigKIg2KrZgtmE2YrZhSDZhNinINmK2YXYs9mRINmG2YLYp9i32KfZiyDZhdit2YXZitipIChnb2xkXyopINmI2YTYpyDZitiu2KfZhNmBINij2LXZhCDYp9mE2YXYs9iq2YjYr9i5ICjZitmP2KjZgtmKINii2K7YsSBOKQogIOKAoiDZhdix2KLYqSBEcml2ZSDYqtit2YXZiiDYo9it2K/YqyDZhtmC2LfYqSDYr9in2KbZhdin2Ysg2K3YqtmJINmE2Ygg2YPYp9mG2Kog2KfZhNi52YXZhNmK2Kkg2KfZhNij2YUg2LPYqtmP2YLYqtmEINio2LnYr9mH2Kcg2KjZhNit2LjYqQoiIiIKaW1wb3J0IG9zCmltcG9ydCBnbG9iCmltcG9ydCBqc29uCmltcG9ydCByZQppbXBvcnQgc2h1dGlsCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKQ0tQVF9QQVRURVJOID0gIkRpVF9lcG9jaF8qX3N0ZXBfKi5wdGgiCgoKZGVmIF9zdGVwX29mKHBhdGgpOgogICAgIiIi2LHZgtmFINin2YTYrti32YjYqSDZhdmGINin2LPZhSDYp9mE2YXZhNmBINin2YTYsdiz2YXZiiBEaVRfZXBvY2hfMDAwMDBfc3RlcF8wMTAwMC5wdGgiIiIKICAgIG0gPSByZS5zZWFyY2gociJzdGVwXyhcZCspIiwgUGF0aChwYXRoKS5zdGVtKQogICAgcmV0dXJuIGludChtLmdyb3VwKDEpKSBpZiBtIGVsc2UgLTEKCgpkZWYgbGlzdF9ja3B0cyhydW5fZGlyKToKICAgICIiItmD2YQg2YbZgtin2Lcg2KfZhNiq2K3ZgtmCINin2YTYsdiz2YXZitipINmB2Yog2YXYrNmE2K8g2KfZhNiq2LTYutmK2YQg2YXYsdiq2KjYqSDYqti12KfYudiv2YrYp9mLINio2KfZhNiu2LfZiNipLiIiIgogICAgcnVuX2RpciA9IFBhdGgocnVuX2RpcikKICAgIHJldHVybiBzb3J0ZWQocnVuX2Rpci5nbG9iKENLUFRfUEFUVEVSTiksIGtleT1fc3RlcF9vZikKCgpkZWYgc2F2ZV9hdG9taWMoc3RhdGUsIHBhdGgpOgogICAgIiIi2K3Zgdi4IHRvcmNoLnNhdmUg2KjYtNmD2YQg2LDYsdmR2Yo6INmK2YPYqtioINil2YTZiSAudG1wINir2YUgcmVuYW1lIOKAlCDZhNinINij2YbYtdin2YEg2YXZhNmB2KfYqiDYo9io2K/Yp9mLLiIiIgogICAgaW1wb3J0IHRvcmNoCiAgICBwYXRoID0gUGF0aChwYXRoKQogICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgdG1wID0gcGF0aC53aXRoX3N1ZmZpeChwYXRoLnN1ZmZpeCArICIudG1wIikKICAgIHRvcmNoLnNhdmUoc3RhdGUsIHRtcCkKICAgICMgZnN5bmMg2LrZitixINmF2KrYp9itINi52KjYsSBBUEkg2KfZhNi52KfZhdipINmE2YAgdG9yY2guc2F2ZdibIHJlbmFtZSDYqNi52K8g2KfZg9iq2YXYp9mEINin2YTZg9iq2KfYqNipINmD2KfZgdmNINi52YXZhNmK2KfZiwogICAgb3MucmVwbGFjZSh0bXAsIHBhdGgpCiAgICByZXR1cm4gcGF0aAoKCmRlZiBwcnVuZV9rZWVwX24ocnVuX2Rpciwga2VlcD0yLCBwcm90ZWN0PSgiZ29sZF8iLCksIGFsc29fZGVsZXRlX2dsb2I9Tm9uZSk6CiAgICAiIiLYqtmC2YTZitmFINii2YXZhjog2YrZj9io2YLZiiDYotiu2LEga2VlcCDZhtmC2LfYqdiMINmI2YrYrdmF2Yog2KPZiiDZhdmE2YEg2YrYqNiv2KMg2KfYs9mF2Ycg2KjYo9it2K8gcHJvdGVjdCBwcmVmaXhlcy4KCiAgICDYo9mK2LbYp9mLOiDZitit2LDZgSDZhdmE2YHYp9iqIC50bXAg2KfZhNmK2KrZitmF2KkgKNmF2YYg2KfZhtmC2LfYp9i5INij2KvZhtin2KEg2KfZhNit2YHYuCkuCiAgICDZiti52YrYryDZgtin2KbZhdipICjYp9mE2YXYrdiw2YjZgdin2KosINin2YTZhdiq2KjZgtmI2YYpLgogICAgIiIiCiAgICBydW5fZGlyID0gUGF0aChydW5fZGlyKQogICAgcmVtb3ZlZCwga2VwdCA9IFtdLCBbXQogICAgIyDYqtmG2LjZitmBINij2YbYtdin2YEg2KfZhNmF2YTZgdin2Kog2KPZiNmE2KfZiwogICAgZm9yIHRtcCBpbiBydW5fZGlyLmdsb2IoIioucHRoLnRtcCIpOgogICAgICAgIHRtcC51bmxpbmsoKQogICAgICAgIHByaW50KGYiW3BydW5lXSDYrdmP2LDZgSDZhdmE2YEg2YbYp9mC2LU6IHt0bXAubmFtZX0iKQogICAgY2twdHMgPSBsaXN0X2NrcHRzKHJ1bl9kaXIpCiAgICBmb3IgYyBpbiBja3B0c1s6LWtlZXBdIGlmIGtlZXAgPiAwIGVsc2UgW106CiAgICAgICAgaWYgYy5uYW1lLnN0YXJ0c3dpdGgocHJvdGVjdCk6CiAgICAgICAgICAgIGtlcHQuYXBwZW5kKGMpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgYy51bmxpbmsoKQogICAgICAgIHJlbW92ZWQuYXBwZW5kKGMubmFtZSkKICAgICAgICBwcmludChmIltwcnVuZV0g2K3Zj9iw2YEge2MubmFtZX0gKNil2KjZgtin2KEg2KLYrtixIHtrZWVwfSkiKQogICAga2VwdCArPSBja3B0c1sta2VlcDpdIGlmIGtlZXAgPiAwIGVsc2UgW10KICAgIHJldHVybiByZW1vdmVkLCBrZXB0CgoKZGVmIG1pcnJvcl90b19kcml2ZShydW5fZGlyLCBkcml2ZV9kaXIsIGtlZXBfb25fZHJpdmU9MiwgcHJvdGVjdD0oImdvbGRfIiwpKToKICAgICIiItmG2LPYriDZhdix2KLYqSDYotmF2YbYqSDYpdmE2YkgRHJpdmU6INmK2Y/Zhtiz2K4g2KfZhNis2K/Zitiv2Iwg2KvZhSDZitmP2YLZhNmO2ZHZhSDYp9mE2YLYr9mK2YUgKNmE2Kcg2YrZj9it2LDZgSDZgtio2YQg2YjYtdmI2YQg2KfZhNio2K/ZitmEKS4KCiAgICBkcml2ZV9kaXI6INmF2LPYp9ixINmF2KzZhNivINin2YTZhtmC2KfYtyDYudmE2YkgRHJpdmUgKNmF2KvYp9mEOiAvY29udGVudC9kcml2ZS9NeURyaXZlL2lyYXFpX3NlZWR2Yy9ydW5zL2lyYXFpX3YxKQogICAgIiIiCiAgICBydW5fZGlyLCBkcml2ZV9kaXIgPSBQYXRoKHJ1bl9kaXIpLCBQYXRoKGRyaXZlX2RpcikKICAgIGlmIG5vdCBkcml2ZV9kaXIuZXhpc3RzKCk6CiAgICAgICAgcHJpbnQoZiJbbWlycm9yXSDZhdis2YTYryBEcml2ZSDYutmK2LEg2YXZiNis2YjYrzoge2RyaXZlX2Rpcn0iKQogICAgICAgIHJldHVybiBbXQogICAgbG9jYWxfY2twdHMgPSBsaXN0X2NrcHRzKHJ1bl9kaXIpCiAgICBpZiBub3QgbG9jYWxfY2twdHM6CiAgICAgICAgcmV0dXJuIFtdCiAgICBjb3BpZWQgPSBbXQogICAgZm9yIGMgaW4gbG9jYWxfY2twdHNbLWtlZXBfb25fZHJpdmU6XToKICAgICAgICBkc3QgPSBkcml2ZV9kaXIgLyBjLm5hbWUKICAgICAgICBpZiBub3QgZHN0LmV4aXN0cygpIG9yIGRzdC5zdGF0KCkuc3Rfc2l6ZSAhPSBjLnN0YXQoKS5zdF9zaXplOgogICAgICAgICAgICB0bXBfZHN0ID0gZHN0LndpdGhfc3VmZml4KCIucHRoLnBhcnQiKQogICAgICAgICAgICBzaHV0aWwuY29weTIoYywgdG1wX2RzdCkKICAgICAgICAgICAgb3MucmVwbGFjZSh0bXBfZHN0LCBkc3QpCiAgICAgICAgICAgIGNvcGllZC5hcHBlbmQoYy5uYW1lKQogICAgICAgICAgICBwcmludChmIlttaXJyb3JdIHtjLm5hbWV9IOKGkiBEcml2ZSAoe2Muc3RhdCgpLnN0X3NpemUvMWU5Oi4yZn1HQikiKQogICAgIyDYqtmC2YTZitmFIERyaXZlINio2LnYryDYp9mE2YbYs9iuICjYrdmF2KfZitipIGdvbGRfINiv2KfYptmF2KfZiykKICAgIGRyaXZlX2NrcHRzID0gbGlzdF9ja3B0cyhkcml2ZV9kaXIpCiAgICBmb3IgYyBpbiBkcml2ZV9ja3B0c1s6LWtlZXBfb25fZHJpdmVdIGlmIGtlZXBfb25fZHJpdmUgPiAwIGVsc2UgW106CiAgICAgICAgaWYgYy5uYW1lLnN0YXJ0c3dpdGgocHJvdGVjdCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgdHJ5OgogICAgICAgICAgICBjLnVubGluaygpCiAgICAgICAgICAgIHByaW50KGYiW21pcnJvcl0g2YLZj9mE2ZDZkdmFINmF2YYgRHJpdmU6IHtjLm5hbWV9IikKICAgICAgICBleGNlcHQgT1NFcnJvciBhcyBlOgogICAgICAgICAgICBwcmludChmIlttaXJyb3JdINiq2LnYsNixINit2LDZgSB7Yy5uYW1lfToge2V9IikKICAgIHJldHVybiBjb3BpZWQKCgpkZWYgd3JpdGVfaGVhcnRiZWF0KHBhdGgsIHBheWxvYWQpOgogICAgIiIi2YbYqNi22Kkg2YLZhNioIEpTT04g2LDYsdmR2YrYqSDigJQg2KrZgtix2KPZh9inINij2K/Yp9ipINin2YTZhdix2KfZgtio2Kkg2YjYp9mE2LPYp9i52Kkg2YTYp9it2YLYp9mLLiIiIgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRtcCA9IHBhdGgud2l0aF9zdWZmaXgoIi50bXAiKQogICAgdG1wLndyaXRlX3RleHQoanNvbi5kdW1wcyhwYXlsb2FkLCBlbnN1cmVfYXNjaWk9RmFsc2UpLCBlbmNvZGluZz0idXRmLTgiKQogICAgb3MucmVwbGFjZSh0bXAsIHBhdGgpCgoKZGVmIHNhdmVfZnVsbF9zdGF0ZShydW5fZGlyLCBzdGF0ZSwgc3RlcCwga2VlcD0yLCBkcml2ZV9kaXI9Tm9uZSwKICAgICAgICAgICAgICAgICAgICBrZWVwX29uX2RyaXZlPTIsIGhlYXJ0YmVhdD1Ob25lLCBwcm90ZWN0PSgiZ29sZF8iLCkpOgogICAgIiIi2KfZhNmI2KfYrNmH2Kkg2KfZhNis2KfZhdi52Kk6INit2YHYuCDYsNix2ZHZiiArINiq2YLZhNmK2YUgKyDZhdix2KLYqSArINmG2KjYttipIOKAlCDYp9iz2KrYr9i52KfYoSDZiNin2K3YryDZhdmGINin2YTZhdiv2LHYqC4KCiAgICBzdGF0ZTog2KfZhNmC2KfZhdmI2LMg2KfZhNix2LPZhdmKINin2YTZg9in2YXZhCB7bmV0LCBvcHRpbWl6ZXIsIHNjaGVkdWxlciwgaXRlcnMsIGVwb2NofQogICAgc3RlcDogINix2YLZhSDYp9mE2K7Yt9mI2Kkg2KfZhNit2KfZhNmK2KkgKNmK2Y/Ys9iq2K7Yr9mFINmE2YTYp9iz2YUg2YjZhNmE2YbYqNi22KkpCiAgICAiIiIKICAgIGVwb2NoID0gc3RhdGUuZ2V0KCJlcG9jaCIsIDApCiAgICBuYW1lID0gZiJEaVRfZXBvY2hfe2Vwb2NoOjA1ZH1fc3RlcF97c3RlcDowNWR9LnB0aCIKICAgIHBhdGggPSBzYXZlX2F0b21pYyhzdGF0ZSwgUGF0aChydW5fZGlyKSAvIG5hbWUpCiAgICBwcmludChmIltja3B0XSDYrdmP2YHYuNiqIHtuYW1lfSIpCiAgICByZW1vdmVkLCBrZXB0ID0gcHJ1bmVfa2VlcF9uKHJ1bl9kaXIsIGtlZXA9a2VlcCwgcHJvdGVjdD1wcm90ZWN0KQogICAgaWYgZHJpdmVfZGlyOgogICAgICAgIG1pcnJvcl90b19kcml2ZShydW5fZGlyLCBkcml2ZV9kaXIsIGtlZXBfb25fZHJpdmU9a2VlcF9vbl9kcml2ZSwgcHJvdGVjdD1wcm90ZWN0KQogICAgaWYgaGVhcnRiZWF0OgogICAgICAgIHdyaXRlX2hlYXJ0YmVhdChoZWFydGJlYXQsIHsKICAgICAgICAgICAgInN0ZXAiOiBpbnQoc3RlcCksICJlcG9jaCI6IGludChlcG9jaCksCiAgICAgICAgICAgICJ0cyI6IHRpbWUudGltZSgpLCAiY2twdCI6IG5hbWUsCiAgICAgICAgICAgICJrZXB0IjogW2submFtZSBmb3IgayBpbiBrZXB0XSwKICAgICAgICB9KQogICAgcmV0dXJuIHBhdGgK",
    "train_iraqi.py": "...",
    "watchdog.py": "IyAtKi0gY29kaW5nOiB1dGYtOCAtKi0KIiIiCndhdGNoZG9nLnB5IOKAlCDYp9mE2K3Yp9ix2LMg2KfZhNiw2YPZiiDZhNmE2KrYr9ix2YrYqCDYp9mE2YXYs9iq2YXYsSDYr9in2K7ZhCDYp9mE2YbZiNin2KkKPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CtmK2LfZhNmCIHRyYWluX2lyYXFpLnB5INmB2YogdGhyZWFkINiu2YTZgdmKICoq2K/Yp9iu2YQg2YbZiNin2KkgQ29sYWIg2YbZgdiz2YfYpyoqICjZhNinIHN1YnByb2Nlc3Mg2LrYsdmK2Kgp2IwK2YHZhNinINiq2Y/Zgtiq2YQg2KfZhNi52YXZhNmK2Kkg2LnZhtivINil2LrZhNin2YIg2KfZhNmF2KrYtdmB2K0g4oCUINin2YTZhtmI2KfYqSDYqtiz2KrZhdixINi32KfZhNmF2Kcg2KfZhNis2YTYs9ipINit2YrYqS4K2KfZhNmF2K/YsdioINin2YTZhdit2LXZkdmGINmK2YjZgtmBINmG2YHYs9mHINmG2LjZitmB2KfZiyDYudmG2K8g2KfZhtiq2YfYp9ihINin2YTZhdmK2LLYp9mG2YrYqdiMINmI2KfZhNit2KfYsdizINmK2LnZitivINil2LfZhNin2YLZhwrZgdmKINis2YTYs9ipIENvbGFiINis2K/Zitiv2Kkg2KrZhNmC2KfYptmK2KfZiyAo2LnZhtivINil2LnYp9iv2Kkg2KrYtNi62YrZhCDYp9mE2K7ZhNmK2KkpLgoK2YTYpyDZitiz2KrYrtiv2YUgc3VicHJvY2Vzcy5Qb3BlbiDZhNmE2KrYr9ix2YrYqCDigJQg2KfZhNiq2K/YsdmK2Kgg2YrYudmF2YQg2KjYp9mE2K7ZitmI2Lcg2K/Yp9iu2YQg2KfZhNi52YXZhNmK2KkK2KfZhNmI2KfYrdiv2KnYjCDZiNin2YTYrdin2LHYsyDZitix2KfZgtioINin2YTYrtmK2LfYjCDZiNmK2LnZitivINin2YTYpdi32YTYp9mCINil2LDYpyDZhdin2Kog2YLYqNmEINio2YTZiNi6INin2YTZh9iv2YEuCgrYp9mE2KfYs9iq2K7Yr9in2YUg2YHZiiDYrtmE2YrYqSBDb2xhYjoKICAgIGZyb20gd2F0Y2hkb2cgaW1wb3J0IHJ1bl90cmFpbmluZ19mb3JldmVyCiAgICBydW5fdHJhaW5pbmdfZm9yZXZlcigKICAgICAgICByZXBvX2Rpcj0iL2NvbnRlbnQvaXJhcWlfc2VlZHZjL3NlZWQtdmMiLAogICAgICAgIGNvbmZpZz0iY29uZmlncy9wcmVzZXRzL2NvbmZpZ19kaXRfbWVsX3NlZWRfdXZpdF93aGlzcGVyX2Jhc2VfZjBfNDRrLnltbCIsCiAgICAgICAgZGF0YXNldF9kaXI9Ii9jb250ZW50L2lyYXFpX3NlZWR2Yy9kYXRhIiwKICAgICAgICBydW5fbmFtZT0iaXJhcWlfdjEiLAogICAgICAgIHRvdGFsX3N0ZXBzPTYwMDAsCiAgICAgICAgdGltZV9idWRnZXRfaG91cnM9MTEsCiAgICAgICAgZHJpdmVfZGlyPSIvY29udGVudC9kcml2ZS9NeURyaXZlL2lyYXFpX3NlZWR2Yy9ydW5zL2lyYXFpX3YxIiwKICAgICkKIiIiCmltcG9ydCBvcwppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmltcG9ydCB0aHJlYWRpbmcKaW1wb3J0IHRpbWUKaW1wb3J0IHRyYWNlYmFjawpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCgpkZWYgX2xvZyhtc2cpOgogICAgcHJpbnQoZiJbd2F0Y2hkb2cge3RpbWUuc3RyZnRpbWUoJyVIOiVNOiVTJyl9XSB7bXNnfSIsIGZsdXNoPVRydWUpCgoKZGVmIHJ1bl90cmFpbmluZ19mb3JldmVyKAogICAgcmVwb19kaXIsCiAgICBjb25maWcsCiAgICBkYXRhc2V0X2RpciwKICAgIHJ1bl9uYW1lPSJpcmFxaV92MSIsCiAgICB0b3RhbF9zdGVwcz02MDAwLAogICAgYmF0Y2hfc2l6ZT0wLAogICAgc2F2ZV9ldmVyeT01MDAsCiAgICBudW1fd29ya2Vycz0yLAogICAgdGltZV9idWRnZXRfaG91cnM9MTEsCiAgICBkcml2ZV9kaXI9Tm9uZSwKICAgIGtlZXBfY2hlY2twb2ludHM9MiwKICAgIGtpdF9kaXI9Tm9uZSwKICAgIG1heF9yZXN0YXJ0cz01MCwKICAgIHZhbGlkYXRlX2ZpcnN0PUZhbHNlLAopOgogICAgIiIi2KXYt9mE2KfZgiDYp9mE2KrYr9ix2YrYqCDZhdi5INmF2LHYp9mC2KjYqSDZiNil2LnYp9iv2Kkg2KrYtNi62YrZhCDYqtmE2YLYp9im2YrYqSDYrdiq2Ykg2KjZhNmI2Log2KfZhNmH2K/ZgSDYp9mE2KrYsdin2YPZhdmKLgoKICAgINmK2LnZhdmEIGJsb2NraW5nINiv2KfYrtmEINin2YTYrtmE2YrYqSAo2YXZgti12YjYryk6INij2LrZhNmC2Kog2KfZhNmF2KrYtdmB2K0g2YHZhNinINmF2LTZg9mE2KnYjCDYp9mE2YbZiNin2Kkg2YXYs9iq2YXYsdipLgogICAg2KPZiNmC2YHZhyDYqNiq2K3YsdmK2LEg2K7Zitin2LEg2KfZhNiq2LTYutmK2YQgKEludGVycnVwdCkg4oCUINin2YTYrdin2LHYsyDZitit2YHYuCDZhtmC2LfYqSDZhti42YrZgdipINi52YbYryDYp9mE2KfZhtmC2LfYp9i5LgogICAgIiIiCiAgICByZXBvX2RpciA9IFBhdGgocmVwb19kaXIpLnJlc29sdmUoKQogICAga2l0X2RpciA9IFBhdGgoa2l0X2RpciBvciBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50KS5yZXNvbHZlKCkKICAgIHNjcmlwdCA9IGtpdF9kaXIgLyAidHJhaW5faXJhcWkucHkiCiAgICBhc3NlcnQgc2NyaXB0LmV4aXN0cygpLCBmInRyYWluX2lyYXFpLnB5INi62YrYsSDZhdmI2KzZiNivINmB2Yoge2tpdF9kaXJ9IgogICAgYXNzZXJ0IChyZXBvX2RpciAvICJ0cmFpbi5weSIpLmV4aXN0cygpLCBmItmF2KzZhNivIHNlZWQtdmMg2LrZitixINi12K3ZititOiB7cmVwb19kaXJ9IgoKICAgIHN5cy5wYXRoLmluc2VydCgwLCBzdHIoa2l0X2RpcikpCiAgICBvcy5lbnZpcm9uLnNldGRlZmF1bHQoIklSQVFJX0tJVF9ESVIiLCBzdHIoa2l0X2RpcikpCgogICAgaWYgdmFsaWRhdGVfZmlyc3Q6CiAgICAgICAgX2xvZygi2YHYrdi1INin2YTYs9mE2KfZhdipINij2YjZhNin2YsgKC0tdmFsaWRhdGUtaW5zdGVhZCkuLi4iKQogICAgICAgIHIgPSBzdWJwcm9jZXNzLnJ1bigKICAgICAgICAgICAgW3N5cy5leGVjdXRhYmxlLCBzdHIoc2NyaXB0KSwgIi0tY29uZmlnIiwgc3RyKGNvbmZpZyksCiAgICAgICAgICAgICAiLS1kYXRhc2V0LWRpciIsIHN0cihkYXRhc2V0X2RpciksICItLXJ1bi1uYW1lIiwgcnVuX25hbWUsCiAgICAgICAgICAgICAiLS12YWxpZGF0ZS1pbnN0ZWFkIl0sCiAgICAgICAgICAgIGN3ZD1zdHIocmVwb19kaXIpLCBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUpCiAgICAgICAgcHJpbnQoci5zdGRvdXQpCiAgICAgICAgaWYgci5yZXR1cm5jb2RlICE9IDA6CiAgICAgICAgICAgIHByaW50KHIuc3RkZXJyKQogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoItmB2LTZhCDZgdit2LUg2KfZhNiz2YTYp9mF2Kkg4oCUINix2KfYrNi5INin2YTZhdiu2LHYrNin2Kog2KPYudmE2KfZhyIpCgogICAgcmVzdGFydHMgPSAwCiAgICB3aGlsZSByZXN0YXJ0cyA8IG1heF9yZXN0YXJ0czoKICAgICAgICAjINij2YrZhiDZiNi12YTZhtin2J8gKNmC2LHYp9ih2Kkg2YbYqNi22Kkg2KfZhNmC2YTYqCDYo9mIINii2K7YsSDZhtmC2LfYqSkKICAgICAgICBpbXBvcnQgY2twdF91dGlscwogICAgICAgIHJ1bl9wYXRoID0gcmVwb19kaXIgLyAicnVucyIgLyBydW5fbmFtZQogICAgICAgIGNrcHRzID0gY2twdF91dGlscy5saXN0X2NrcHRzKHJ1bl9wYXRoKQogICAgICAgIHJlYWNoZWQgPSBja3B0X3V0aWxzLl9zdGVwX29mKGNrcHRzWy0xXSkgaWYgY2twdHMgZWxzZSAwCiAgICAgICAgaWYgcmVhY2hlZCA+PSB0b3RhbF9zdGVwczoKICAgICAgICAgICAgX2xvZyhmItin2YTZh9iv2YEg2KfZhNiq2LHYp9mD2YXZiiB7dG90YWxfc3RlcHN9INmF2K3ZgtmCICjZiNi12YTZhtinIHtyZWFjaGVkfSkg4oCUINin2YbYqtmH2Ykg2KfZhNi52YXZhCDinJMiKQogICAgICAgICAgICByZXR1cm4KICAgICAgICByZW1haW5pbmcgPSB0b3RhbF9zdGVwcyAtIHJlYWNoZWQKICAgICAgICBfbG9nKGYi2KfZhNmI2LbYuToge3JlYWNoZWR9L3t0b3RhbF9zdGVwc30gfCDZhdiq2KjZgtmNIHtyZW1haW5pbmd9IHwg2KXYt9mE2KfZgiAje3Jlc3RhcnRzICsgMX0iKQoKICAgICAgICBjbWQgPSBbc3lzLmV4ZWN1dGFibGUsICItdSIsIHN0cihzY3JpcHQpLAogICAgICAgICAgICAgICAiLS1jb25maWciLCBzdHIoY29uZmlnKSwKICAgICAgICAgICAgICAgIi0tZGF0YXNldC1kaXIiLCBzdHIoZGF0YXNldF9kaXIpLAogICAgICAgICAgICAgICAiLS1ydW4tbmFtZSIsIHJ1bl9uYW1lLAogICAgICAgICAgICAgICAiLS10b3RhbC1zdGVwcyIsIHN0cih0b3RhbF9zdGVwcyksCiAgICAgICAgICAgICAgICItLXNhdmUtZXZlcnkiLCBzdHIoc2F2ZV9ldmVyeSksCiAgICAgICAgICAgICAgICItLW51bS13b3JrZXJzIiwgc3RyKG51bV93b3JrZXJzKSwKICAgICAgICAgICAgICAgIi0tdGltZS1idWRnZXQtaG91cnMiLCBzdHIodGltZV9idWRnZXRfaG91cnMpLAogICAgICAgICAgICAgICAiLS1rZWVwLWNoZWNrcG9pbnRzIiwgc3RyKGtlZXBfY2hlY2twb2ludHMpXQogICAgICAgIGlmIGJhdGNoX3NpemU6CiAgICAgICAgICAgIGNtZCArPSBbIi0tYmF0Y2gtc2l6ZSIsIHN0cihiYXRjaF9zaXplKV0KICAgICAgICBpZiBkcml2ZV9kaXI6CiAgICAgICAgICAgIGNtZCArPSBbIi0tZHJpdmUtZGlyIiwgc3RyKGRyaXZlX2RpcildCgogICAgICAgICMgLS0tLS0tLS0tLSDYp9mE2KrYtNi62YrZhCDYr9in2K7ZhCDYp9mE2LnZhdmE2YrYqSAo2K7Ziti3KSAtLS0tLS0tLS0tCiAgICAgICAgZXhpdF9jb2RlID0gTm9uZQoKICAgICAgICBkZWYgX3J1bigpOgogICAgICAgICAgICBub25sb2NhbCBleGl0X2NvZGUKICAgICAgICAgICAgIyDYqti02LrZitmEINin2YTYqtiv2LHZitioINmB2Yog2YfYsNmHINin2YTYudmF2YTZitipOiBzeXMuYXJndiDYq9mFIGV4ZWMgKNin2YTZhdmE2YEg2YXZg9iq2YjYqCDYqNij2LPZhNmI2KggX19tYWluX18pCiAgICAgICAgICAgIG9zLmNoZGlyKHJlcG9fZGlyKQogICAgICAgICAgICAjIGNtZCA9IFtweXRob24sIC11LCBzY3JpcHQsIGFyZ3MuLi5dIOKGkiBzeXMuYXJndiA9IFtzY3JpcHQsIGFyZ3MuLi5dCiAgICAgICAgICAgIHN5cy5hcmd2ID0gW3N0cihzY3JpcHQpXSArIGNtZFszOl0KICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgY29kZSA9IHNjcmlwdC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikKICAgICAgICAgICAgICAgIG5zID0geyJfX25hbWVfXyI6ICJfX21haW5fXyIsICJfX2ZpbGVfXyI6IHN0cihzY3JpcHQpfQogICAgICAgICAgICAgICAgZXhlYyhjb21waWxlKGNvZGUsIHN0cihzY3JpcHQpLCAiZXhlYyIpLCBucykKICAgICAgICAgICAgICAgIGV4aXRfY29kZSA9IDAKICAgICAgICAgICAgZXhjZXB0IFN5c3RlbUV4aXQgYXMgZToKICAgICAgICAgICAgICAgIGV4aXRfY29kZSA9IGUuY29kZSBpZiBpc2luc3RhbmNlKGUuY29kZSwgaW50KSBlbHNlIDAKICAgICAgICAgICAgZXhjZXB0IEJhc2VFeGNlcHRpb246CiAgICAgICAgICAgICAgICB0cmFjZWJhY2sucHJpbnRfZXhjKCkKICAgICAgICAgICAgICAgIGV4aXRfY29kZSA9IDEKCiAgICAgICAgdCA9IHRocmVhZGluZy5UaHJlYWQodGFyZ2V0PV9ydW4sIGRhZW1vbj1GYWxzZSkKICAgICAgICB0LnN0YXJ0KCkKICAgICAgICB0LmpvaW4oKQoKICAgICAgICAjINio2LnYryDYp9mG2KrZh9in2KEg2KfZhNiu2YrYtzog2KPZitmGINmI2LXZhNmG2Kcg2KfZhNii2YbYnwogICAgICAgIHRpbWUuc2xlZXAoNSkKICAgICAgICBja3B0cyA9IGNrcHRfdXRpbHMubGlzdF9ja3B0cyhydW5fcGF0aCkKICAgICAgICByZWFjaGVkID0gY2twdF91dGlscy5fc3RlcF9vZihja3B0c1stMV0pIGlmIGNrcHRzIGVsc2UgMAogICAgICAgIGlmIHJlYWNoZWQgPj0gdG90YWxfc3RlcHM6CiAgICAgICAgICAgIF9sb2coZiLYp9mE2YfYr9mBINmF2K3ZgtmCICh7cmVhY2hlZH0ve3RvdGFsX3N0ZXBzfSkg4oCUINin2YbYqtmH2Ykg2KfZhNi52YXZhCDinJMiKQogICAgICAgICAgICByZXR1cm4KCiAgICAgICAgcmVzdGFydHMgKz0gMQogICAgICAgIGlmIGV4aXRfY29kZSBub3QgaW4gKDAsIE5vbmUpOgogICAgICAgICAgICBfbG9nKGYi2KfZhtiq2YfZiSDYp9mE2KrYtNi62YrZhCDYqNix2YXYsiB7ZXhpdF9jb2RlfSDigJQg2KXYudin2K/YqSDYp9mE2KXYt9mE2KfZgiDYqNi52K8gNjAg2KvYp9mG2YrYqS4uLiIpCiAgICAgICAgICAgIHRpbWUuc2xlZXAoNjApCiAgICAgICAgZWxzZToKICAgICAgICAgICAgX2xvZygi2KfZhtiq2YfZiSDYp9mE2KrYtNi62YrZhCAo2YXZitiy2KfZhtmK2Kkg2LnZhNmJINin2YTYo9ix2KzYrSkg4oCUINil2LnYp9iv2Kkg2KfZhNil2LfZhNin2YIg2YHZiNix2KfZiyDZgdmKINin2YTYrNmE2LPYqSDZhtmB2LPZh9inINil2YYg2KPZhdmD2YYiKQogICAgICAgICAgICB0aW1lLnNsZWVwKDEwKQoKICAgIF9sb2coZiLYqtis2KfZiNiy2YbYpyDYrdivINil2LnYp9iv2Kkg2KfZhNiq2LTYutmK2YQgKHttYXhfcmVzdGFydHN9KSDigJQg2LTYutmR2YQg2KfZhNiu2YTZitipINmF2KzYr9iv2KfZiyDZhNmF2KrYp9io2LnYqSDYp9mE2YfYr9mBIikKCgpkZWYgY2hlY2tfc2Vzc2lvbl9oZWFsdGgoZHJpdmVfZGlyPU5vbmUsIGhvdXJzPTI0KToKICAgICIiItmB2K3YtSDYs9ix2YrYuTog2YXYp9iw2Kcg2YrZiNis2K8g2YXYrdmE2YrYp9mLINmI2LnZhNmJIERyaXZl2J8g2YTYt9io2KfYudiq2Ycg2YHZiiDYqNiv2KfZitipINij2Yog2KzZhNiz2KkuIiIiCiAgICBpbXBvcnQgY2twdF91dGlscwogICAgcHJpbnQoIj0iICogNjApCiAgICBwcmludCgi2YHYrdi1INit2KfZhNipINin2YTYrNmE2LPYqSIpCiAgICBwcmludCgiPSIgKiA2MCkKICAgIGZvciBsYWJlbCwgYmFzZSBpbiAoKCLZhdit2YTZiiIsIFBhdGgoInJ1bnMiKSksICgiRHJpdmUiLCBQYXRoKGRyaXZlX2RpcikgaWYgZHJpdmVfZGlyIGVsc2UgTm9uZSkpOgogICAgICAgIGlmIGJhc2UgaXMgTm9uZSBvciBub3QgYmFzZS5leGlzdHMoKToKICAgICAgICAgICAgcHJpbnQoZiJbe2xhYmVsfV0g2LrZitixINmF2YjYrNmI2K8iKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGZvciBydW4gaW4gc29ydGVkKGJhc2UuZ2xvYigiKi8iKSk6CiAgICAgICAgICAgIGNrcHRzID0gY2twdF91dGlscy5saXN0X2NrcHRzKHJ1bikKICAgICAgICAgICAgaGIgPSBydW4gLyAiaGVhcnRiZWF0Lmpzb24iCiAgICAgICAgICAgIGhiX2luZm8gPSAiIgogICAgICAgICAgICBpZiBoYi5leGlzdHMoKToKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBpbXBvcnQganNvbiBhcyBfanNvbgogICAgICAgICAgICAgICAgICAgIGggPSBfanNvbi5sb2FkcyhoYi5yZWFkX3RleHQoKSkKICAgICAgICAgICAgICAgICAgICBoYl9pbmZvID0gKGYiIHwg2YbYqNi22Kk6INiu2LfZiNipIHtoLmdldCgnc3RlcCcpfSDZgtio2YQgIgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZiJ7aW50KHRpbWUudGltZSgpLWguZ2V0KCd0cycsMCkpLy82MH0g2K/ZgtmK2YLYqSIpCiAgICAgICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgICAgIHBhc3MKICAgICAgICAgICAgaWYgY2twdHM6CiAgICAgICAgICAgICAgICBwcmludChmIlt7bGFiZWx9XSB7cnVuLm5hbWV9OiB7bGVuKGNrcHRzKX0g2YbZgti32KnYjCDYo9it2K/Yq9mH2Kcge2NrcHRzWy0xXS5uYW1lfXtoYl9pbmZvfSIpCiAgICAgICAgICAgIGVsaWYgKHJ1biAvICJmdF9tb2RlbC5wdGgiKS5leGlzdHMoKToKICAgICAgICAgICAgICAgIHByaW50KGYiW3tsYWJlbH1dIHtydW4ubmFtZX06IGZ0X21vZGVsLnB0aCDZgdmC2LciKQogICAgcHJpbnQoIj0iICogNjApCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIGltcG9ydCBhcmdwYXJzZQogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcmVwby1kaXIiLCByZXF1aXJlZD1UcnVlKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWNvbmZpZyIsIHJlcXVpcmVkPVRydWUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZGF0YXNldC1kaXIiLCByZXF1aXJlZD1UcnVlKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXJ1bi1uYW1lIiwgZGVmYXVsdD0iaXJhcWlfdjEiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRvdGFsLXN0ZXBzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NjAwMCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS10aW1lLWJ1ZGdldC1ob3VycyIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MTEpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZHJpdmUtZGlyIiwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXZhbGlkYXRlLWZpcnN0IiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIGEgPSBhcC5wYXJzZV9hcmdzKCkKICAgIHJ1bl90cmFpbmluZ19mb3JldmVyKAogICAgICAgIHJlcG9fZGlyPWEucmVwb19kaXIsIGNvbmZpZz1hLmNvbmZpZywgZGF0YXNldF9kaXI9YS5kYXRhc2V0X2RpciwKICAgICAgICBydW5fbmFtZT1hLnJ1bl9uYW1lLCB0b3RhbF9zdGVwcz1hLnRvdGFsX3N0ZXBzLAogICAgICAgIHRpbWVfYnVkZ2V0X2hvdXJzPWEudGltZV9idWRnZXRfaG91cnMsIGRyaXZlX2Rpcj1hLmRyaXZlX2RpciwKICAgICAgICB2YWxpZGF0ZV9maXJzdD1hLnZhbGlkYXRlX2ZpcnN0LAogICAgKQo=",
    "loss_monitor.py": "IyAtKi0gY29kaW5nOiB1dGYtOCAtKi0KIiIiCmxvc3NfbW9uaXRvci5weSDigJQg2YXYsdin2YLYqCDYqtiv2LHZitioIFNlZWQtVkMg2YHZiiDYp9mE2LLZhdmGINin2YTYrdmC2YrZgtmKICjYudix2KjZiikKPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQrZitmC2LHYoyB0cmFpbl9jb25zb2xlLmxvZyAo2KPZiCBsb3NzX2hpc3RvcnkuanNvbmwpINmI2YrYudix2LY6CiAg4oCiINmF2YbYrdmG2Ykg2KfZhNiu2LPYp9ix2Kkg2K/Yp9iu2YQg2KfZhNmG2YjYp9ipIChtYXRwbG90bGliKSArINii2K7YsSDYp9mE2YLZitmFCiAg4oCiINin2YTYs9ix2LnYqSDYp9mE2YHYudmE2YrYqSDZiNin2YTYpdiz2YLYp9i3INin2YTYstmF2YbZiiDYrdiq2Ykg2KfZhNmH2K/ZgQogIOKAoiDYrdmD2YUg2LXYrdipOiDYp9iz2KrZgtix2KfYsS/ZhdmK2YQvTmFOICsg2KrZgtix2YrYsSDZg9in2YXZhCDZgtin2KjZhCDZhNmE2YbYs9iuCiAg4oCiINmF2LHYp9mC2KjYqSDZhdiz2KrZhdix2Kk6IGxvc3NfbW9uaXRvci5weSAtLXdhdGNoIExPRwoK2KfZhNin2LPYqtiu2K/Yp9mFOgogICAgcHl0aG9uIGxvc3NfbW9uaXRvci5weSBydW5fZGlyLyAgICAgICAgICAgICMg2YLYsdin2KHYqSBsb3NzX2hpc3RvcnkuanNvbmwKICAgIHB5dGhvbiBsb3NzX21vbml0b3IucHkgdHJhaW5fY29uc29sZS5sb2cgICAjINiq2K3ZhNmK2YQg2LPYrNmEINmG2LXZigogICAgcHl0aG9uIGxvc3NfbW9uaXRvci5weSBMT0cgLS13YXRjaCAgICAgICAgICMg2KrYrdiv2YrYqyDZg9mEIDYwINir2KfZhtmK2KkKICAgIHB5dGhvbiBsb3NzX21vbml0b3IucHkgTE9HIC0tdGFyZ2V0IDYwMDAgLS1idWRnZXQtaG91cnMgMTEKIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgcmUKaW1wb3J0IHN0YXRpc3RpY3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKClNURVBfUkUgPSByZS5jb21waWxlKHIiZXBvY2hccysoXGQrKVxzKixccypzdGVwXHMrKFxkKylccyosXHMqbG9zczpccyooWzAtOS5lRSstXSspIikKVFFETV9SRSA9IHJlLmNvbXBpbGUociIoXGQrKS8oXGQrKVxzKlxbW15cXV0qPyxccyooWzAtOS5dKykocy9pdHxpdC9zKSIpCkgxMiA9IDEyICogMzYwMAoKCmRlZiBwYXJzZV9jb25zb2xlKHRleHQpOgogICAgcHRzLCBzaXRzLCBlcG9jaF9sZW4gPSBbXSwgW10sIE5vbmUKICAgIGZvciBsaW5lIGluIHRleHQuc3BsaXRsaW5lcygpOgogICAgICAgIG0gPSBTVEVQX1JFLnNlYXJjaChsaW5lKQogICAgICAgIGlmIG06CiAgICAgICAgICAgIHB0cy5hcHBlbmQoKGludChtLmdyb3VwKDIpKSwgZmxvYXQobS5ncm91cCgzKSkpKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIG0gPSBUUURNX1JFLnNlYXJjaChsaW5lKQogICAgICAgIGlmIG06CiAgICAgICAgICAgIGlmIGVwb2NoX2xlbiBpcyBOb25lOgogICAgICAgICAgICAgICAgZXBvY2hfbGVuID0gaW50KG0uZ3JvdXAoMikpCiAgICAgICAgICAgIHYgPSBmbG9hdChtLmdyb3VwKDMpKQogICAgICAgICAgICBzaXRzLmFwcGVuZCh2IGlmIG0uZ3JvdXAoNCkgPT0gInMvaXQiIGVsc2UgMS4wIC8gdikKICAgIHJldHVybiBwdHMsIHNpdHMsIGVwb2NoX2xlbgoKCmRlZiBwYXJzZV9oaXN0b3J5KHBhdGgpOgogICAgcHRzID0gW10KICAgIGZvciBsaW5lIGluIFBhdGgocGF0aCkucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpLnNwbGl0bGluZXMoKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIGQgPSBqc29uLmxvYWRzKGxpbmUpCiAgICAgICAgICAgIHB0cy5hcHBlbmQoKGludChkWyJzdGVwIl0pLCBmbG9hdChkWyJlbWFfbG9zcyJdKSkpCiAgICAgICAgZXhjZXB0IChqc29uLkpTT05EZWNvZGVFcnJvciwgS2V5RXJyb3IsIFZhbHVlRXJyb3IpOgogICAgICAgICAgICBjb250aW51ZQogICAgcmV0dXJuIHB0cwoKCmRlZiBzbG9wZV9wZXJfMTAwMChwdHMpOgogICAgaWYgbGVuKHB0cykgPCAzOgogICAgICAgIHJldHVybiAwLjAKICAgIHhzID0gW3BbMF0gZm9yIHAgaW4gcHRzXQogICAgeXMgPSBbcFsxXSBmb3IgcCBpbiBwdHNdCiAgICBteCwgbXkgPSBzdGF0aXN0aWNzLm1lYW4oeHMpLCBzdGF0aXN0aWNzLm1lYW4oeXMpCiAgICBudW0gPSBzdW0oKHggLSBteCkgKiAoeSAtIG15KSBmb3IgeCwgeSBpbiB6aXAoeHMsIHlzKSkKICAgIGRlbiA9IHN1bSgoeCAtIG14KSAqKiAyIGZvciB4IGluIHhzKQogICAgcmV0dXJuIChudW0gLyBkZW4pICogMTAwMC4wIGlmIGRlbiBlbHNlIDAuMAoKCmRlZiBhbmFseXplKHB0cywgc2l0cywgZXBvY2hfbGVuLCB0YXJnZXQsIGJ1ZGdldF9ob3Vycyk6CiAgICBpZiBub3QgcHRzOgogICAgICAgIHJldHVybiBOb25lLCAi2YTYpyDZhtmC2KfYtyDYrtiz2KfYsdipINio2LnYry4iCiAgICB2YWxzID0gW3YgZm9yIF8sIHYgaW4gcHRzXQogICAgc3RlcHMgPSBbcyBmb3IgcywgXyBpbiBwdHNdCiAgICBrID0gbWF4KGxlbih2YWxzKSAvLyAxMCwgMSkKICAgIHNpdCA9IHN0YXRpc3RpY3MubWVkaWFuKHNpdHMpIGlmIHNpdHMgZWxzZSBOb25lCiAgICByID0gewogICAgICAgICJuIjogbGVuKHZhbHMpLCAiZmlyc3QiOiBzdGVwc1swXSwgImxhc3QiOiBzdGVwc1stMV0sCiAgICAgICAgIm1lYW4iOiBzdGF0aXN0aWNzLm1lYW4odmFscyksCiAgICAgICAgInN0ZCI6IHN0YXRpc3RpY3Muc3RkZXYodmFscykgaWYgbGVuKHZhbHMpID4gMSBlbHNlIDAuMCwKICAgICAgICAibWluIjogbWluKHZhbHMpLCAibWF4IjogbWF4KHZhbHMpLAogICAgICAgICJzbG9wZSI6IHNsb3BlX3Blcl8xMDAwKHB0cyksCiAgICAgICAgImZpcnN0X2siOiBzdGF0aXN0aWNzLm1lYW4odmFsc1s6a10pLCAibGFzdF9rIjogc3RhdGlzdGljcy5tZWFuKHZhbHNbLWs6XSksCiAgICAgICAgIm5hbiI6IGFueSh2ICE9IHYgZm9yIHYgaW4gdmFscyksICAjIG5vcWE6IEY2MzIKICAgICAgICAic2l0Ijogc2l0LCAiZXBvY2hfbGVuIjogZXBvY2hfbGVuLAogICAgfQogICAgbGluZXMgPSBbXQogICAgbGluZXMuYXBwZW5kKCI9IiAqIDYyKQogICAgbGluZXMuYXBwZW5kKCLZhdix2KfZgtioINin2YTYqtiv2LHZitioIOKAlCBTZWVkLVZDINin2YTYudix2KfZgtmKIikKICAgIGxpbmVzLmFwcGVuZCgiPSIgKiA2MikKICAgIGxpbmVzLmFwcGVuZChmItin2YTZhtmC2KfYtzoge3JbJ24nXX0gfCDYp9mE2YXYr9mJOiDYrti32YjYqSB7clsnZmlyc3QnXX0g4oaSIHtyWydsYXN0J119IikKICAgIGxpbmVzLmFwcGVuZChmItin2YTZhdiq2YjYs9i3IMKxINin2YTYp9mG2K3Ysdin2YEgOiB7clsnbWVhbiddOi40Zn0gwrEge3JbJ3N0ZCddOi40Zn0iKQogICAgbGluZXMuYXBwZW5kKGYi2KfZhNij2K/ZhtmJIC8g2KfZhNij2LnZhNmJICAgICA6IHtyWydtaW4nXTouNGZ9IC8ge3JbJ21heCddOi40Zn0iKQogICAgbGluZXMuYXBwZW5kKGYi2KfZhNmF2YrZhCDZhNmD2YQgMTAwMCDYrti32YjYqToge3JbJ3Nsb3BlJ106Ky40Zn0iKQogICAgbGluZXMuYXBwZW5kKGYi2KPZiNmEIDEwJSDihpIg2KLYrtixIDEwJSAgOiB7clsnZmlyc3RfayddOi40Zn0g4oaSIHtyWydsYXN0X2snXTouNGZ9IikKICAgIGlmIHJbIm5hbiJdOgogICAgICAgIGxpbmVzLmFwcGVuZCgi4pqgINiq2YjYrNivINmC2YrZhSBOYU4g2YHZiiDYp9mE2LPYrNmEISIpCiAgICBpZiBzaXQ6CiAgICAgICAgbGluZXMuYXBwZW5kKGYi2KfZhNiz2LHYudipINin2YTZiNiz2YrYt9ipICAgICA6IHtzaXQ6LjJmfSDYqy/Yrti32YjYqSAoezEwMDAqc2l0LzYwOi4wZn0g2K/ZgtmK2YLYqS8xMDAwINiu2LfZiNipKSIpCiAgICAgICAgaWYgdGFyZ2V0OgogICAgICAgICAgICByZW0gPSBtYXgodGFyZ2V0IC0gclsibGFzdCJdLCAwKQogICAgICAgICAgICBocnMgPSByZW0gKiBzaXQgLyAzNjAwCiAgICAgICAgICAgIGZpdCA9ICLinJMg2K/Yp9iu2YQg2KzZhNiz2KkgMTIg2LPYp9i52KkiIGlmIHJlbSAqIHNpdCA8PSBIMTIgZWxzZSAi4pyXINiq2KrYrNin2YjYsiDYrNmE2LPYqSDZiNin2K3Yr9ipIgogICAgICAgICAgICBsaW5lcy5hcHBlbmQoZiLYrdiq2Ykge3RhcmdldH06INmF2KrYqNmC2Y0ge3JlbX0g4omIIHtocnM6LjFmfSDYs9in2LnYqSDigJQge2ZpdH0iKQogICAgICAgIGlmIGJ1ZGdldF9ob3VyczoKICAgICAgICAgICAgZml0ID0gKGYi4pyTINmK2YPZgdmKINi22YXZhiDZhdmK2LLYp9mG2YrYqSB7YnVkZ2V0X2hvdXJzfSDYs9in2LnYqSIKICAgICAgICAgICAgICAgICAgIGlmICh0YXJnZXQgLSByWydsYXN0J10pICogc2l0IDw9IGJ1ZGdldF9ob3VycyAqIDM2MDAKICAgICAgICAgICAgICAgICAgIGVsc2UgZiLinJcg2YTZhiDZitmD2KrZhdmEINi22YXZhiB7YnVkZ2V0X2hvdXJzfSDYs9in2LnYqSDigJQg2YLZhNmR2LUg2KfZhNmH2K/ZgSDYo9mIINmB2LnZkdmEINin2YTYrNmE2LPYp9iqINin2YTZhdiq2LnYr9iv2KkiKQogICAgICAgICAgICBsaW5lcy5hcHBlbmQoZiLYp9mE2YXZitiy2KfZhtmK2Kk6IHtmaXR9IikKICAgIGlmIHJbImVwb2NoX2xlbiJdOgogICAgICAgIGxpbmVzLmFwcGVuZChmIti32YjZhCDYp9mE2K3Zgtio2Kk6IHtyWydlcG9jaF9sZW4nXX0g2K/Zgdi52Kkg4omIIHtyWydlcG9jaF9sZW4nXSpzaXQvMzYwMDouMWZ9INiz2KfYudipIiBpZiBzaXQgZWxzZSAiIikKICAgIHZlcmRpY3QgPSAoItiz2YTZitmFIOKAlCDYp9iz2KrZhdix2Jsg2KfZhNix2YLZhSBFTUEg2YXZhti52ZHZhSAwLjk5INmI2KfZhNmC2LHYp9ixINin2YTZhtmH2KfYptmKINmE2YTYqtmC2YrZitmFINin2YTZhdmI2LbZiNi52YoiCiAgICAgICAgICAgICAgIGlmIG5vdCByWyJuYW4iXSBhbmQgclsibWVhbiJdIDwgMS4wIGFuZCByWyJzbG9wZSJdIDwgMC4wMgogICAgICAgICAgICAgICBlbHNlICLZitiz2KrZiNis2Kgg2KfZhtiq2KjYp9mH2KfZiyDigJQg2LHYp9mC2Kgg2KfZhNiq2YLZitmK2YUg2KfZhNmF2YjYttmI2LnZiiDYudmG2K8g2KPZgtix2Kgg2YbZgti32KkiKQogICAgbGluZXMuYXBwZW5kKCItIiAqIDYyKQogICAgbGluZXMuYXBwZW5kKGYi2KfZhNit2YPZhToge3ZlcmRpY3R9IikKICAgIGxpbmVzLmFwcGVuZCgiPSIgKiA2MikKICAgIHJldHVybiByLCAiXG4iLmpvaW4obGluZXMpCgoKZGVmIHRyeV9wbG90KHB0cywgc2F2ZV90bz1Ob25lKToKICAgIHRyeToKICAgICAgICBpbXBvcnQgbWF0cGxvdGxpYgogICAgICAgIG1hdHBsb3RsaWIudXNlKCJBZ2ciKQogICAgICAgIGltcG9ydCBtYXRwbG90bGliLnB5cGxvdCBhcyBwbHQKICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICBwcmludCgiW3Bsb3RdIG1hdHBsb3RsaWIg2LrZitixINmF2KrYp9itIOKAlCDYudix2LYg2YbYtdmKINmB2YLYtyIpCiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICB4cyA9IFtzIGZvciBzLCBfIGluIHB0c10KICAgIHlzID0gW3YgZm9yIF8sIHYgaW4gcHRzXQogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg4LCAzLjIpLCBjb25zdHJhaW5lZF9sYXlvdXQ9VHJ1ZSkKICAgIGF4LnBsb3QoeHMsIHlzLCBsdz0xLjUsIGNvbG9yPSIjNEM2RUY1IikKICAgIG1lYW4gPSBzdGF0aXN0aWNzLm1lYW4oeXMpCiAgICBheC5heGhsaW5lKG1lYW4sIGxzPSItLSIsIGx3PTEsIGNvbG9yPSIjQzY4NjZBIiwgbGFiZWw9ZiJtZWFuIHttZWFuOi40Zn0iKQogICAgYXguc2V0X3hsYWJlbCgic3RlcCIpCiAgICBheC5zZXRfeWxhYmVsKCJFTUEgbG9zcyIpCiAgICBheC5zZXRfdGl0bGUoIlNlZWQtVkMgSXJhcWkgVHJhaW5pbmciKQogICAgYXguZ3JpZChhbHBoYT0wLjIpCiAgICBheC5sZWdlbmQoKQogICAgaWYgc2F2ZV90bzoKICAgICAgICBmaWcuc2F2ZWZpZyhzYXZlX3RvLCBkcGk9MTUwKQogICAgICAgIHBsdC5jbG9zZShmaWcpCiAgICAgICAgcHJpbnQoZiJbcGxvdF0g2K3Zj9mB2Lgg2KfZhNmF2YbYrdmG2Yk6IHtzYXZlX3RvfSIpCiAgICBlbHNlOgogICAgICAgIHBsdC5zaG93KCkKICAgIHJldHVybiBUcnVlCgoKZGVmIG1haW4oKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249ItmF2LHYp9mC2Kgg2KrYr9ix2YrYqCBTZWVkLVZDICjYudix2KjZiikiKQogICAgYXAuYWRkX2FyZ3VtZW50KCJzcmMiLCBoZWxwPSLZhdmE2YEg2KfZhNiz2KzZhCDYo9mIINmF2KzZhNivINin2YTYqti02LrZitmEICjZitio2K3YqyDYudmGIGxvc3NfaGlzdG9yeS5qc29ubCkiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRhcmdldCIsIHR5cGU9aW50LCBkZWZhdWx0PTYwMDApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYnVkZ2V0LWhvdXJzIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXdhdGNoIiwgYWN0aW9uPSJzdG9yZV90cnVlIiwgaGVscD0i2YXYsdin2YLYqNipINmF2LPYqtmF2LHYqSDZg9mEIDYwINir2KfZhtmK2KkiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXBsb3QiLCBkZWZhdWx0PU5vbmUsIGhlbHA9Itit2YHYuCDYp9mE2YXZhtit2YbZiSBQTkcg2KXZhNmJINmH2LDYpyDYp9mE2YXYs9in2LEiKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQoKICAgIGRlZiBvbmNlKCk6CiAgICAgICAgc3JjID0gUGF0aChhcmdzLnNyYykKICAgICAgICBpZiBzcmMuaXNfZGlyKCk6CiAgICAgICAgICAgIGhpc3QgPSBzcmMgLyAibG9zc19oaXN0b3J5Lmpzb25sIgogICAgICAgICAgICBjb25zb2xlID0gc3JjIC8gInRyYWluX2NvbnNvbGUubG9nIgogICAgICAgICAgICBpZiBoaXN0LmV4aXN0cygpOgogICAgICAgICAgICAgICAgcHRzID0gcGFyc2VfaGlzdG9yeShoaXN0KQogICAgICAgICAgICAgICAgc2l0cywgZXBvY2hfbGVuID0gW10sIE5vbmUKICAgICAgICAgICAgZWxpZiBjb25zb2xlLmV4aXN0cygpOgogICAgICAgICAgICAgICAgdGV4dCA9IGNvbnNvbGUucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIsIGVycm9ycz0icmVwbGFjZSIpCiAgICAgICAgICAgICAgICBwdHMsIHNpdHMsIGVwb2NoX2xlbiA9IHBhcnNlX2NvbnNvbGUodGV4dCkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHByaW50KGYi2YTYpyDYs9is2YQg2YHZiiB7c3JjfSAo2KjYrdir2Kog2LnZhiBsb3NzX2hpc3RvcnkuanNvbmwg2YggdHJhaW5fY29uc29sZS5sb2cpIikKICAgICAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgZWxzZToKICAgICAgICAgICAgaWYgc3JjLnN1ZmZpeCA9PSAiLmpzb25sIjoKICAgICAgICAgICAgICAgIHB0cyA9IHBhcnNlX2hpc3Rvcnkoc3JjKQogICAgICAgICAgICAgICAgc2l0cywgZXBvY2hfbGVuID0gW10sIE5vbmUKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHRleHQgPSBzcmMucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIsIGVycm9ycz0icmVwbGFjZSIpCiAgICAgICAgICAgICAgICBwdHMsIHNpdHMsIGVwb2NoX2xlbiA9IHBhcnNlX2NvbnNvbGUodGV4dCkKICAgICAgICByLCByZXBvcnQgPSBhbmFseXplKHB0cywgc2l0cywgZXBvY2hfbGVuLCBhcmdzLnRhcmdldCwgYXJncy5idWRnZXRfaG91cnMpCiAgICAgICAgaWYgcmVwb3J0OgogICAgICAgICAgICBwcmludChyZXBvcnQpCiAgICAgICAgaWYgYXJncy5wbG90IGFuZCBwdHM6CiAgICAgICAgICAgIHRyeV9wbG90KHB0cywgYXJncy5wbG90KQogICAgICAgIHJldHVybiByCgogICAgaWYgYXJncy53YXRjaDoKICAgICAgICBwcmludCgiW3dhdGNoXSDZhdix2KfZgtio2Kkg2YXYs9iq2YXYsdipIOKAlCBDdHJsK0Mg2YTZhNil2YrZgtin2YEiKQogICAgICAgIHRyeToKICAgICAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgICAgIHByaW50KCJcbiIgKyAiIyIgKiA2MikKICAgICAgICAgICAgICAgIG9uY2UoKQogICAgICAgICAgICAgICAgdGltZS5zbGVlcCg2MCkKICAgICAgICBleGNlcHQgS2V5Ym9hcmRJbnRlcnJ1cHQ6CiAgICAgICAgICAgIHByaW50KCJcblt3YXRjaF0g2KPZj9mI2YLZgdiqINin2YTZhdix2KfZgtio2KkiKQogICAgZWxzZToKICAgICAgICBvbmNlKCkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==",
    "evaluate_ckpt.py": "IyAtKi0gY29kaW5nOiB1dGYtOCAtKi0KIiIiCmV2YWx1YXRlX2NrcHQucHkg4oCUINin2YTYqtmC2YrZitmFINin2YTZhdmI2LbZiNi52Yog2YTZhtmC2KfYtyBTZWVkLVZDICjYqti02KfYqNmHINmF2KrYrdiv2KsgKyDYt9io2YrYudmK2KkpCj09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CtmK2YLZitmR2YUg2YbZgtin2Lcg2KfZhNiq2K3ZgtmCINi52YTZiSDYudmK2YbYp9iqINin2K7Yqtio2KfYsSDZhdi52LLZiNmE2Kkg2LnYqNixIGluZmVyZW5jZS5weSDYp9mE2LHYs9mF2Yog2KvZhSDZitit2LPYqDoKICDigKIgU0lNOiDYqti02KfYqNmHINin2YTZhdiq2K3Yr9irICjYqti22YXZitmGIENBTVBQbHVz2Iwg2KzZitioINin2YTYqtmF2KfZhSkg4oCUINin2YTZhdmC2YrYp9izINin2YTYo9iz2KfYs9mKCiAg4oCiIFVUTU9TOiDYqtmI2YLYuSBNT1Mg2YTZhNi32KjZiti52YrYqSAoMeKAkzXYjCDYqtix2KrZitioINmG2LPYqNmKINmB2YLYtyDZhNij2YbZhyDYr9mP2LHZkdioINi52YTZiSBFTi9aSCkKICDigKIg2K7YtyDYo9iz2KfYsyDYpdmE2LLYp9mF2Yo6INin2YTZhtmF2YjYsNisINin2YTYsdiz2YXZiiDYutmK2LEg2KfZhNmF2LbYqNmI2Lcg2LnZhNmJINmG2YHYsyDYp9mE2LnZitmG2KfYqgoK2KfZhNmC2LHYp9ixOiDYp9mE2YbZgti32Kkg2KfZhNij2YHYttmEID0g2KPYudmE2YkgU0lNINmF2LkgVVRNT1Mg2YTYpyDZitmG2YfYp9ixICjZhNinINmK2YbYrtmB2LYgPiAwLjIg2LnZhiDYrti3INin2YTYo9iz2KfYsykuCgrYp9mE2KfYs9iq2K7Yr9in2YUgKNmF2YYg2K/Yp9iu2YQg2YXYrNmE2K8gc2VlZC12Yyk6CiAgICBweXRob24gZXZhbHVhdGVfY2twdC5weSAtLXJ1bi1kaXIgcnVucy9pcmFxaV92MSAtLXRlc3QtZGlyIC9wYXRoL3RvL2hvbGRvdXQgXAogICAgICAgIC0tY29uZmlnIHJ1bnMvaXJhcWlfdjEvY29uZmlnXyoueW1sIC0tZHJpdmUtZGlyIC9jb250ZW50L2RyaXZlLy4uLi9pcmFxaV92MQogICAgIyDYpdi22KfZgdipINmG2YLYt9ipINmK2K/ZiNmK2Kk6IC0tZXh0cmEtY2twdCBydW5zL2lyYXFpX3YxL2dvbGRfc3RhZ2VfQS5wdGgKIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgY3N2CmltcG9ydCBnbG9iCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgcmUKaW1wb3J0IHNodXRpbAppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmltcG9ydCBzdGF0aXN0aWNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKQVVESU9fRVhUID0geyIud2F2IiwgIi5tcDMiLCAiLmZsYWMiLCAiLm9nZyIsICIub3B1cyIsICIubTRhIn0KCgpkZWYgX3N0ZXBfb2YocCk6CiAgICBtID0gcmUuc2VhcmNoKHIic3RlcF8oXGQrKSIsIFBhdGgocCkuc3RlbSkKICAgIHJldHVybiBpbnQobS5ncm91cCgxKSkgaWYgbSBlbHNlIC0xCgoKZGVmIGZpbmRfY29uZmlnKHJ1bl9kaXIpOgogICAgY2FuZHMgPSBzb3J0ZWQoUGF0aChydW5fZGlyKS5nbG9iKCJjb25maWdfKi55bWwiKSkKICAgIGlmIG5vdCBjYW5kczoKICAgICAgICAjINix2KjZhdinIC55YW1sINij2Ygg2KfZhNil2LnYr9in2K8g2YHZiiDZhdis2YTYryDYp9mE2KXYudiv2KfYryDYp9mE2KPYtdmE2Yog4oCUINin2KjYrdirINij2LnZhdmCCiAgICAgICAgY2FuZHMgPSBzb3J0ZWQoUGF0aChydW5fZGlyKS5nbG9iKCIqLnltbCIpKQogICAgYXNzZXJ0IGNhbmRzLCBmItmE2Kcg2YXZhNmBINil2LnYr9in2K/Yp9iqINmB2Yoge3J1bl9kaXJ9IgogICAgcmV0dXJuIGNhbmRzWzBdCgoKZGVmIHBpY2tfc2FtcGxlcyh0ZXN0X2Rpciwgbl9zb3VyY2VzPTMsIG5fcmVmcz0yLCByZWZzX2Rpcj1Ob25lKToKICAgIHNyY3MgPSBzb3J0ZWQoW3AgZm9yIHAgaW4gUGF0aCh0ZXN0X2RpcikuaXRlcmRpcigpIGlmIHAuc3VmZml4Lmxvd2VyKCkgaW4gQVVESU9fRVhUXSkKICAgIGFzc2VydCBzcmNzLCBmItmE2Kcg2YXZgtin2LfYuSDYp9iu2KrYqNin2LEg2YHZiiB7dGVzdF9kaXJ9IgogICAgc291cmNlcyA9IHNyY3NbOm5fc291cmNlc10KICAgIGlmIHJlZnNfZGlyIGFuZCBQYXRoKHJlZnNfZGlyKS5leGlzdHMoKToKICAgICAgICByZWZzID0gc29ydGVkKFtwIGZvciBwIGluIFBhdGgocmVmc19kaXIpLml0ZXJkaXIoKSBpZiBwLnN1ZmZpeC5sb3dlcigpIGluIEFVRElPX0VYVF0pWzpuX3JlZnNdCiAgICBlbHNlOgogICAgICAgIHJlZnMgPSBzcmNzWy1uX3JlZnM6XSBpZiBsZW4oc3JjcykgPj0gbl9yZWZzICsgbl9zb3VyY2VzIGVsc2Ugc291cmNlc1s6bl9yZWZzXQogICAgcmV0dXJuIHNvdXJjZXMsIHJlZnMKCgpkZWYgY29udmVydF9vbmUoc3JjLCByZWYsIG91dF9kaXIsIGNvbmZpZywgY2twdD1Ob25lLCBkaWZmdXNpb25fc3RlcHM9MzAsIHB5dGhvbj1Ob25lKToKICAgICIiItiq2LTYutmK2YQgaW5mZXJlbmNlLnB5INin2YTYsdiz2YXZiiDZhNmG2YLYt9ipINmI2KfYrdiv2KkuINmK2LnZitivINmF2LPYp9ixINin2YTZhdiu2LHYrCDYo9mIIE5vbmUuIiIiCiAgICBjbWQgPSBbcHl0aG9uIG9yIHN5cy5leGVjdXRhYmxlLCAiaW5mZXJlbmNlLnB5IiwKICAgICAgICAgICAiLS1zb3VyY2UiLCBzdHIoc3JjKSwgIi0tdGFyZ2V0Iiwgc3RyKHJlZiksCiAgICAgICAgICAgIi0tb3V0cHV0Iiwgc3RyKG91dF9kaXIpLCAiLS1jb25maWciLCBzdHIoY29uZmlnKSwKICAgICAgICAgICAiLS1kaWZmdXNpb24tc3RlcHMiLCBzdHIoZGlmZnVzaW9uX3N0ZXBzKSwKICAgICAgICAgICAiLS1mMC1jb25kaXRpb24iLCAiVHJ1ZSIsICItLWF1dG8tZjAtYWRqdXN0IiwgIlRydWUiLAogICAgICAgICAgICItLWZwMTYiLCAiVHJ1ZSJdCiAgICBpZiBja3B0IGlzIG5vdCBOb25lOgogICAgICAgIGNtZCArPSBbIi0tY2hlY2twb2ludCIsIHN0cihja3B0KV0KICAgIHIgPSBzdWJwcm9jZXNzLnJ1bihjbWQsIGNhcHR1cmVfb3V0cHV0PVRydWUsIHRleHQ9VHJ1ZSkKICAgICMg2KfZhNiq2LPZhdmK2Kkg2KfZhNix2LPZhdmK2Kk6IHZjX3tzb3VyY2V9X3t0YXJnZXR9X3tsZW5ndGhfYWRqdXN0fV97c3RlcHN9X3tjZmd9LndhdgogICAgZXhwZWN0ZWQgPSBOb25lCiAgICBmb3IgZiBpbiBQYXRoKG91dF9kaXIpLmdsb2IoInZjXyoud2F2Iik6CiAgICAgICAgaWYgUGF0aChzcmMpLnN0ZW0gaW4gZi5uYW1lIGFuZCBQYXRoKHJlZikuc3RlbSBpbiBmLm5hbWU6CiAgICAgICAgICAgIGV4cGVjdGVkID0gZgogICAgICAgICAgICBicmVhawogICAgaWYgci5yZXR1cm5jb2RlICE9IDAgb3IgZXhwZWN0ZWQgaXMgTm9uZToKICAgICAgICBwcmludChmIltjb252ZXJ0XSDZgdi02YQge1BhdGgoc3JjKS5uYW1lfcOXe1BhdGgocmVmKS5uYW1lfSIKICAgICAgICAgICAgICBmInsnICjZhtmC2LfYqSAnICsgUGF0aChja3B0KS5uYW1lICsgJyknIGlmIGNrcHQgZWxzZSAnJ30iKQogICAgICAgIHByaW50KChyLnN0ZG91dCBvciAiIilbLTQwMDpdLCAoci5zdGRlcnIgb3IgIiIpWy00MDA6XSkKICAgICAgICByZXR1cm4gTm9uZQogICAgcmV0dXJuIGV4cGVjdGVkCgoKZGVmIGNhbXBwbHVzX3NpbSh3YXYxLCB3YXYyLCBzdl9tb2RlbCwgZGV2aWNlKToKICAgIGltcG9ydCB0b3JjaAogICAgaW1wb3J0IHRvcmNoYXVkaW8uY29tcGxpYW5jZS5rYWxkaSBhcyBrYWxkaQogICAgaW1wb3J0IGxpYnJvc2EKCiAgICBkZWYgZW1iZWQocGF0aCk6CiAgICAgICAgdywgXyA9IGxpYnJvc2EubG9hZChzdHIocGF0aCksIHNyPTE2MDAwLCBtb25vPVRydWUpCiAgICAgICAgZiA9IGthbGRpLmZiYW5rKHRvcmNoLmZyb21fbnVtcHkodykudW5zcXVlZXplKDApLCBudW1fbWVsX2JpbnM9ODAsCiAgICAgICAgICAgICAgICAgICAgICAgIGRpdGhlcj0wLCBzYW1wbGVfZnJlcXVlbmN5PTE2MDAwKQogICAgICAgIGYgPSBmIC0gZi5tZWFuKGRpbT0wLCBrZWVwZGltPVRydWUpCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgIHJldHVybiBzdl9tb2RlbChmLnVuc3F1ZWV6ZSgwKS50byhkZXZpY2UpKS5zcXVlZXplKDApCgogICAgcmV0dXJuIGZsb2F0KHRvcmNoLm5uLmZ1bmN0aW9uYWwuY29zaW5lX3NpbWlsYXJpdHkoCiAgICAgICAgZW1iZWQod2F2MSksIGVtYmVkKHdhdjIpLCBkaW09MCkuaXRlbSgpKQoKCmRlZiB1dG1vc19zY29yZSh3YXZfcGF0aCwgcHJlZGljdG9yLCBkZXZpY2UpOgogICAgaW1wb3J0IHRvcmNoCiAgICBpbXBvcnQgbGlicm9zYQogICAgdywgXyA9IGxpYnJvc2EubG9hZChzdHIod2F2X3BhdGgpLCBzcj0xNjAwMCwgbW9ubz1UcnVlKQogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgcmV0dXJuIGZsb2F0KHByZWRpY3Rvcih0b3JjaC5mcm9tX251bXB5KHcpLnVuc3F1ZWV6ZSgwKS50byhkZXZpY2UpLCAxNjAwMCkuaXRlbSgpKQoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSLYp9mE2KrZgtmK2YrZhSDYp9mE2YXZiNi22YjYudmKINmE2YbZgtin2LcgU2VlZC1WQyIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcnVuLWRpciIsIHJlcXVpcmVkPVRydWUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdGVzdC1kaXIiLCByZXF1aXJlZD1UcnVlLCBoZWxwPSLZhdmC2KfYt9i5INin2YTYp9iu2KrYqNin2LEg2KfZhNmF2LnYstmI2YTYqSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY29uZmlnIiwgZGVmYXVsdD1Ob25lLCBoZWxwPSLYp9mB2KrYsdin2LbZitin2Ys6IGNvbmZpZ18qLnltbCDZhdmGIHJ1bi1kaXIiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRyaXZlLWRpciIsIGRlZmF1bHQ9Tm9uZSwgaGVscD0i2YbYs9iuINin2YTZhtiq2KfYptisINil2YTZiSBEcml2ZSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcmVmcy1kaXIiLCBkZWZhdWx0PU5vbmUsIGhlbHA9Itij2LXZiNin2Kog2YXYsdis2LnZitipINio2K/ZitmE2KkgKNin2YHYqtix2KfYttmKOiDZhdmGINi52YrZhtin2Kog2KfZhNin2K7Yqtio2KfYsSkiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1heC1ja3B0cyIsIHR5cGU9aW50LCBkZWZhdWx0PTYpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZXh0cmEtY2twdCIsIGFjdGlvbj0iYXBwZW5kIiwgZGVmYXVsdD1bXSwKICAgICAgICAgICAgICAgICAgICBoZWxwPSLZhtmC2LfYqSDYpdi22KfZgdmK2Kkg2YTZhNiq2YLZitmK2YUgKNmF2YPYsdixKSDigJQg2YXYq9in2YQg2YbZgtin2LcgZ29sZF8qIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1uLXNvdXJjZXMiLCB0eXBlPWludCwgZGVmYXVsdD0zKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW4tcmVmcyIsIHR5cGU9aW50LCBkZWZhdWx0PTIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZGlmZnVzaW9uLXN0ZXBzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MzApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2tpcC1iYXNlbGluZSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0td29ya2RpciIsIGRlZmF1bHQ9Tm9uZSwgaGVscD0i2KfZgdiq2LHYp9i22YrYp9mLIDxydW4tZGlyPi8uLi9ldmFsX3J1biIpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgcnVuX2RpciA9IFBhdGgoYXJncy5ydW5fZGlyKS5yZXNvbHZlKCkKICAgIHRlc3RfZGlyID0gUGF0aChhcmdzLnRlc3RfZGlyKS5yZXNvbHZlKCkKICAgIGFzc2VydCBydW5fZGlyLmV4aXN0cygpLCBmItmF2KzZhNivINin2YTYqti02LrZitmEINi62YrYsSDZhdmI2KzZiNivOiB7cnVuX2Rpcn0iCiAgICBhc3NlcnQgdGVzdF9kaXIuZXhpc3RzKCksIGYi2YXYrNmE2K8g2KfZhNin2K7Yqtio2KfYsSDYutmK2LEg2YXZiNis2YjYrzoge3Rlc3RfZGlyfSIKCiAgICBjb25maWcgPSBQYXRoKGFyZ3MuY29uZmlnKSBpZiBhcmdzLmNvbmZpZyBlbHNlIGZpbmRfY29uZmlnKHJ1bl9kaXIpCiAgICB3b3JrZGlyID0gUGF0aChhcmdzLndvcmtkaXIpIGlmIGFyZ3Mud29ya2RpciBlbHNlIHJ1bl9kaXIucGFyZW50IC8gImV2YWxfcnVuIgogICAgd29ya2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgIyDYo9it2K/YqyBOINmG2YLYp9i3ICsg2YbZgtin2Lcg2KXYttin2YHZitipICjZhtiy2Lkg2KfZhNiq2YPYsdin2LEpCiAgICBja3B0cyA9IHNvcnRlZChydW5fZGlyLmdsb2IoIkRpVF9lcG9jaF8qX3N0ZXBfKi5wdGgiKSwga2V5PV9zdGVwX29mKVstYXJncy5tYXhfY2twdHM6XQogICAgZm9yIGUgaW4gYXJncy5leHRyYV9ja3B0OgogICAgICAgIGVwID0gUGF0aChlKQogICAgICAgIGlmIGVwLmV4aXN0cygpIGFuZCBlcCBub3QgaW4gY2twdHM6CiAgICAgICAgICAgIGNrcHRzLmFwcGVuZChlcCkKICAgIGFzc2VydCBja3B0cywgItmE2Kcg2YbZgtin2Lcg2KrZgtmK2YrZhSDigJQg2LTYutmR2YQg2KfZhNiq2K/YsdmK2Kgg2KPZiNmE2KfZiyIKCiAgICBzb3VyY2VzLCByZWZzID0gcGlja19zYW1wbGVzKHRlc3RfZGlyLCBhcmdzLm5fc291cmNlcywgYXJncy5uX3JlZnMsIGFyZ3MucmVmc19kaXIpCiAgICBwcmludChmItin2YTZhtmC2KfYtyDYp9mE2YXYsdi02K3YqSAoe2xlbihja3B0cyl9KToiKQogICAgZm9yIGMgaW4gY2twdHM6CiAgICAgICAgcHJpbnQoZiIgIOKAoiB7Yy5uYW1lfSIpCiAgICBwcmludChmItin2YTZhdi12KfYr9ixOiB7W3MubmFtZSBmb3IgcyBpbiBzb3VyY2VzXX0iKQogICAgcHJpbnQoZiLYp9mE2YXYsdin2KzYuToge1tyLm5hbWUgZm9yIHIgaW4gcmVmc119XG4iKQoKICAgICMgLS0tLS0tLS0tLSDYp9mE2YXYsdit2YTYqSAxOiDYp9mE2KrYrdmI2YrZhCDYudio2LEgaW5mZXJlbmNlLnB5INin2YTYsdiz2YXZiiAtLS0tLS0tLS0tCiAgICBqb2JzID0gW10KICAgIGlmIG5vdCBhcmdzLnNraXBfYmFzZWxpbmU6CiAgICAgICAgam9icy5hcHBlbmQoKE5vbmUsICJiYXNlbGluZV9vZmZpY2lhbCIpKQogICAgam9icyArPSBbKGMsIGMuc3RlbSkgZm9yIGMgaW4gY2twdHNdCgogICAgZm9yIGNrcHQsIHRhZyBpbiBqb2JzOgogICAgICAgIG91dCA9IHdvcmtkaXIgLyB0YWcKICAgICAgICBvdXQubWtkaXIoZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBpZiAob3V0IC8gIi5kb25lIikuZXhpc3RzKCk6CiAgICAgICAgICAgIHByaW50KGYiW9iq2K7Yt9mR2YpdIHt0YWd9IOKAlCDZhdmP2K3ZiNmO2ZHZhNipINmF2LPYqNmC2KfZiyIpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgb2sgPSAwCiAgICAgICAgZm9yIHNyYyBpbiBzb3VyY2VzOgogICAgICAgICAgICBmb3IgcmVmIGluIHJlZnM6CiAgICAgICAgICAgICAgICBpZiBjb252ZXJ0X29uZShzcmMsIHJlZiwgb3V0LCBjb25maWcsIGNrcHQsIGFyZ3MuZGlmZnVzaW9uX3N0ZXBzKToKICAgICAgICAgICAgICAgICAgICBvayArPSAxCiAgICAgICAgcHJpbnQoZiJbY29udmVydF0ge3RhZ306IHtva30ve2xlbihzb3VyY2VzKSpsZW4ocmVmcyl9INmF2K7Ysdis2KfYqiIpCiAgICAgICAgKG91dCAvICIuZG9uZSIpLnRvdWNoKCkKCiAgICAjIC0tLS0tLS0tLS0g2KfZhNmF2LHYrdmE2KkgMjog2KfZhNmF2YLYp9mK2YrYsyAtLS0tLS0tLS0tCiAgICBwcmludCgiXG7Yqtit2YXZitmEINmF2YLYp9mK2YrYsyDYp9mE2KrZgtmK2YrZhS4uLiIpCiAgICBpbXBvcnQgdG9yY2gKICAgIGZyb20gbW9kdWxlcy5jYW1wcGx1cy5EVEROTiBpbXBvcnQgQ0FNUFBsdXMKICAgIGZyb20gaGZfdXRpbHMgaW1wb3J0IGxvYWRfY3VzdG9tX21vZGVsX2Zyb21faGYKCiAgICBkZXZpY2UgPSAiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiCiAgICBzdiA9IENBTVBQbHVzKGZlYXRfZGltPTgwLCBlbWJlZGRpbmdfc2l6ZT0xOTIpCiAgICBiaW5fcGF0aCA9IGxvYWRfY3VzdG9tX21vZGVsX2Zyb21faGYoImZ1bmFzci9jYW1wcGx1cyIsICJjYW1wcGx1c19jbl9jb21tb24uYmluIiwgTm9uZSkKICAgIHN2LmxvYWRfc3RhdGVfZGljdCh0b3JjaC5sb2FkKGJpbl9wYXRoLCBtYXBfbG9jYXRpb249ImNwdSIpKQogICAgc3YuZXZhbCgpLnRvKGRldmljZSkKCiAgICB1dG1vcyA9IE5vbmUKICAgIHRyeToKICAgICAgICB1dG1vcyA9IHRvcmNoLmh1Yi5sb2FkKCJ0YXJlcGFuL1NwZWVjaE1PUzp2MS4yLjAiLCAidXRtb3MyMl9zdHJvbmciLCB0cnVzdF9yZXBvPVRydWUpLnRvKGRldmljZSkKICAgICAgICBwcmludCgiICBVVE1PUzIyIOKckyAodG9yY2guaHViKSIpCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgcHJpbnQoZiIgIFVUTU9TINi62YrYsSDZhdiq2KfYrSAoe2V9KSDigJQgU0lNINmB2YLYtyIpCgogICAgcm93cywgc3VtbWFyeSA9IFtdLCB7fQogICAgZm9yIGNrcHQsIHRhZyBpbiBqb2JzOgogICAgICAgIG91dCA9IHdvcmtkaXIgLyB0YWcKICAgICAgICBzaW1zLCB1dG1zID0gW10sIFtdCiAgICAgICAgZm9yIHNyYyBpbiBzb3VyY2VzOgogICAgICAgICAgICBmb3IgcmVmIGluIHJlZnM6CiAgICAgICAgICAgICAgICBnZW4gPSBOb25lCiAgICAgICAgICAgICAgICBmb3IgZiBpbiBvdXQuZ2xvYigidmNfKi53YXYiKToKICAgICAgICAgICAgICAgICAgICBpZiBzcmMuc3RlbSBpbiBmLm5hbWUgYW5kIHJlZi5zdGVtIGluIGYubmFtZToKICAgICAgICAgICAgICAgICAgICAgICAgZ2VuID0gZgogICAgICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICAgICAgaWYgZ2VuIGlzIE5vbmU6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBzaW0gPSBjYW1wcGx1c19zaW0oZ2VuLCByZWYsIHN2LCBkZXZpY2UpCiAgICAgICAgICAgICAgICAgICAgc2ltcy5hcHBlbmQoc2ltKQogICAgICAgICAgICAgICAgICAgIHJvdyA9IHsiY2twdCI6IHRhZywgInNyYyI6IHNyYy5uYW1lLCAicmVmIjogcmVmLm5hbWUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICJzaW0iOiByb3VuZChzaW0sIDQpfQogICAgICAgICAgICAgICAgICAgIGlmIHV0bW9zIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAgICAgICB1ID0gdXRtb3Nfc2NvcmUoZ2VuLCB1dG1vcywgZGV2aWNlKQogICAgICAgICAgICAgICAgICAgICAgICB1dG1zLmFwcGVuZCh1KQogICAgICAgICAgICAgICAgICAgICAgICByb3dbInV0bW9zIl0gPSByb3VuZCh1LCAzKQogICAgICAgICAgICAgICAgICAgIHJvd3MuYXBwZW5kKHJvdykKICAgICAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgICAgICAgICBwcmludChmIlttZWFzdXJlXSDYrti32KMge3RhZ30ve3NyYy5uYW1lfToge2V9IikKICAgICAgICBpZiBzaW1zOgogICAgICAgICAgICBzdW1tYXJ5W3RhZ10gPSB7CiAgICAgICAgICAgICAgICAic2ltX21lYW4iOiByb3VuZChzdGF0aXN0aWNzLm1lYW4oc2ltcyksIDQpLAogICAgICAgICAgICAgICAgInNpbV9zdGQiOiByb3VuZChzdGF0aXN0aWNzLnN0ZGV2KHNpbXMpLCA0KSBpZiBsZW4oc2ltcykgPiAxIGVsc2UgMC4wLAogICAgICAgICAgICAgICAgInV0bW9zX21lYW4iOiByb3VuZChzdGF0aXN0aWNzLm1lYW4odXRtcyksIDMpIGlmIHV0bXMgZWxzZSBOb25lLAogICAgICAgICAgICAgICAgIm4iOiBsZW4oc2ltcyksCiAgICAgICAgICAgIH0KCiAgICAjIC0tLS0tLS0tLS0g2KfZhNmF2LHYrdmE2KkgMzog2KfZhNiq2LHYqtmK2Kgg2YjYp9mE2YLYsdin2LEgLS0tLS0tLS0tLQogICAgcHJpbnQoIlxuIiArICI9IiAqIDY0KQogICAgcHJpbnQoZiJ7J9in2YTZhtmC2LfYqSc6PDQwfSB7J1NJTSc6Pjd9IHsnwrEnOj42fSB7J1VUTU9TJzo+N30geyduJzo+M30iKQogICAgcHJpbnQoIi0iICogNjQpCiAgICBmb3IgdGFnLCB2IGluIHNvcnRlZChzdW1tYXJ5Lml0ZW1zKCksIGtleT1sYW1iZGEga3Y6IC1rdlsxXVsic2ltX21lYW4iXSk6CiAgICAgICAgcHJpbnQoZiJ7dGFnOjw0MH0ge3ZbJ3NpbV9tZWFuJ106PjcuNGZ9IHt2WydzaW1fc3RkJ106PjYuNGZ9ICIKICAgICAgICAgICAgICBmInt2Wyd1dG1vc19tZWFuJ10gaWYgdlsndXRtb3NfbWVhbiddIGlzIG5vdCBOb25lIGVsc2UgJy0nOj43fSB7dlsnbiddOj4zfSIpCiAgICBwcmludCgiPSIgKiA2NCkKCiAgICBiYXNlbGluZSA9IHN1bW1hcnkuZ2V0KCJiYXNlbGluZV9vZmZpY2lhbCIpCiAgICBpZiBiYXNlbGluZToKICAgICAgICBiZXN0X3RhZyA9IG1heCgodCBmb3IgdCBpbiBzdW1tYXJ5IGlmIHQgIT0gImJhc2VsaW5lX29mZmljaWFsIiksCiAgICAgICAgICAgICAgICAgICAgICAga2V5PWxhbWJkYSB0OiBzdW1tYXJ5W3RdWyJzaW1fbWVhbiJdLCBkZWZhdWx0PU5vbmUpCiAgICAgICAgaWYgYmVzdF90YWc6CiAgICAgICAgICAgIGdhaW4gPSBzdW1tYXJ5W2Jlc3RfdGFnXVsic2ltX21lYW4iXSAtIGJhc2VsaW5lWyJzaW1fbWVhbiJdCiAgICAgICAgICAgIHZlcmRpY3QgPSAoIuKckyDYqtit2LPZkdmGINit2YLZitmC2Yog2LnZhiDYp9mE2LHYs9mF2YoiIGlmIGdhaW4gPj0gMC4wMiBlbHNlCiAgICAgICAgICAgICAgICAgICAgICAgIuKJiCDZgtix2YrYqCDZhdmGINin2YTYsdiz2YXZiiDigJQg2YLZitmR2YUg2KjYp9mE2LPZhdi5IiBpZiBnYWluID49IDAgZWxzZQogICAgICAgICAgICAgICAgICAgICAgICLinJcg2YTZhSDZitiq2YHZiNmCINio2LnYryDigJQg2KrYp9io2Lkg2KfZhNiq2K/YsdmK2KgiKQogICAgICAgICAgICBwcmludChmIlxu2K7YtyDYp9mE2KPYs9in2LMgU0lNPXtiYXNlbGluZVsnc2ltX21lYW4nXTouNGZ9IHwgIgogICAgICAgICAgICAgICAgICBmItin2YTYo9mB2LbZhCB7YmVzdF90YWd9IFNJTT17c3VtbWFyeVtiZXN0X3RhZ11bJ3NpbV9tZWFuJ106LjRmfSAiCiAgICAgICAgICAgICAgICAgIGYiKNmB2LHZgiB7Z2FpbjorLjRmfSkg4oCUIHt2ZXJkaWN0fSIpCgogICAgIyAtLS0tLS0tLS0tINin2YTYrdmB2LggLS0tLS0tLS0tLQogICAgY3N2X3BhdGggPSB3b3JrZGlyIC8gImV2YWxfcmVzdWx0cy5jc3YiCiAgICB3aXRoIG9wZW4oY3N2X3BhdGgsICJ3IiwgbmV3bGluZT0iIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICB3ID0gY3N2LkRpY3RXcml0ZXIoZiwgZmllbGRuYW1lcz1bImNrcHQiLCAic3JjIiwgInJlZiIsICJzaW0iLCAidXRtb3MiXSkKICAgICAgICB3LndyaXRlaGVhZGVyKCkKICAgICAgICB3LndyaXRlcm93cyhyb3dzKQogICAgKHdvcmtkaXIgLyAiZXZhbF9zdW1tYXJ5Lmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgIGpzb24uZHVtcHMoc3VtbWFyeSwgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MiksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBwcmludChmIlxuW9mG2KrYp9im2KxdIHtjc3ZfcGF0aH0iKQoKICAgIGlmIGFyZ3MuZHJpdmVfZGlyOgogICAgICAgIGRydiA9IFBhdGgoYXJncy5kcml2ZV9kaXIpCiAgICAgICAgZHJ2Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBzaHV0aWwuY29weTIoY3N2X3BhdGgsIGRydiAvICJldmFsX3Jlc3VsdHMuY3N2IikKICAgICAgICBzaHV0aWwuY29weTIod29ya2RpciAvICJldmFsX3N1bW1hcnkuanNvbiIsIGRydiAvICJldmFsX3N1bW1hcnkuanNvbiIpCiAgICAgICAgcHJpbnQoZiJbRHJpdmVdINmG2Y/Ys9iu2Kog2KfZhNmG2KrYp9im2Kwg2KXZhNmJIHtkcnZ9IikKCiAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBzeXMuZXhpdChtYWluKCkpCg==",
    "data_utils.py": "IyAtKi0gY29kaW5nOiB1dGYtOCAtKi0KIiIiCmRhdGFfdXRpbHMucHkg4oCUINij2K/ZiNin2Kog2KfZhNio2YrYp9mG2KfYqiDZiNmG2YLYp9i3INin2YTYqtit2YLZgiDZhNmAIFNlZWQtVkMg2KfZhNi52LHYp9mC2YoKPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQogIGF1ZGl0KCkgICAgICAgICAgOiDYqtiv2YLZitmCINi02KfZhdmEINmE2YXYrNmE2K8g2KjZitin2YbYp9iqICjYudiv2K8v2YXYr9ipL9i12YrYui/Ys9in2LnYp9iqL9mF2KrYrdiv2KvZiNmGINiq2YLYsdmK2KjZitin2YspCiAgbWFrZV9ob2xkb3V0KCkgICA6INi52LLZhCDZhdmC2KfYt9i5INin2K7Yqtio2KfYsSDZhdi52LLZiNmE2Kkg2LnZhiDYp9mE2KrYr9ix2YrYqCAo2YXYuSDYrdmF2KfZitipINmF2YYg2KXYudin2K/YqSDYp9mE2KrYs9ix2YrYqCkKICB2ZXJpZnlfY2twdCgpICAgIDog2YHYrdi1INiz2YTYp9mF2Kkg2KPZiiDZhtmC2LfYqSAo2YXZgdin2KrZititICsg2KPZiNiy2KfZhiBmaW5pdGUgKyDYrdin2YTYqSDZhdit2LPZkdmGICsg2K7Yt9mI2KkpCiAgZmluZ2VycHJpbnQoKSAgICA6INio2LXZhdipINiz2LHZiti52Kkg2YTZhdis2YTYryDYp9mE2KjZitin2YbYp9iqINmE2YXZhti5INiq2LPYsdmR2Kgg2KfZhNin2K7Yqtio2KfYsQoK2KfYs9iq2K7Yr9in2YUgQ0xJOgogICAgcHl0aG9uIGRhdGFfdXRpbHMucHkgYXVkaXQgL3BhdGgvdG8vZGF0YQogICAgcHl0aG9uIGRhdGFfdXRpbHMucHkgaG9sZG91dCAvcGF0aC90by9kYXRhIC0tY291bnQgMjQgLS10ZXN0LWRpciAvcGF0aC90by9ob2xkb3V0CiAgICBweXRob24gZGF0YV91dGlscy5weSB2ZXJpZnkgcnVucy9pcmFxaV92MS9EaVRfZXBvY2hfMDAwMDBfc3RlcF8wMTAwMC5wdGgKIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgcmFuZG9tCmltcG9ydCBzaHV0aWwKaW1wb3J0IHN5cwpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKQVVESU9fRVhUID0geyIud2F2IiwgIi5tcDMiLCAiLmZsYWMiLCAiLm9nZyIsICIub3B1cyIsICIubTRhIn0KTUlOX0RVUiwgTUFYX0RVUiA9IDEuMCwgMzAuMAoKCmRlZiBpdGVyX2F1ZGlvKGRhdGFfZGlyKToKICAgIGZvciBwIGluIFBhdGgoZGF0YV9kaXIpLnJnbG9iKCIqIik6CiAgICAgICAgaWYgcC5zdWZmaXgubG93ZXIoKSBpbiBBVURJT19FWFQ6CiAgICAgICAgICAgIHlpZWxkIHAKCgpkZWYgYXVkaW9fZHVyYXRpb24ocGF0aCk6CiAgICAiIiLZhdiv2Kkg2KfZhNmF2YLYt9i5INio2KfZhNir2YjYp9mG2YogKNio2YTYpyDYqtit2YXZitmEINmD2KfZhdmEINmE2YTYqNmK2KfZhtin2Kog2LnZhtivINin2YTYpdmF2YPYp9mGKS4iIiIKICAgIHRyeToKICAgICAgICBpbXBvcnQgc291bmRmaWxlIGFzIHNmCiAgICAgICAgaW5mbyA9IHNmLmluZm8oc3RyKHBhdGgpKQogICAgICAgIHJldHVybiBmbG9hdChpbmZvLmZyYW1lcyAvIGluZm8uc2FtcGxlcmF0ZSkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgdHJ5OgogICAgICAgICAgICBpbXBvcnQgbGlicm9zYQogICAgICAgICAgICByZXR1cm4gZmxvYXQobGlicm9zYS5nZXRfZHVyYXRpb24ocGF0aD1zdHIocGF0aCkpKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHJldHVybiBOb25lCgoKZGVmIGZpbmdlcnByaW50KGRhdGFfZGlyKToKICAgICIiItio2LXZhdipIFNIQTEg2YTYo9iz2YXYp9ihINmI2KPYrdis2KfZhSDZg9mEINin2YTZhdmC2KfYt9i5IOKAlCDZhNin2YPYqti02KfZgSDYo9mKINiq2LrZitmR2LEg2YHZiiDYp9mE2KjZitin2YbYp9iqLiIiIgogICAgaCA9IGhhc2hsaWIuc2hhMSgpCiAgICBmb3IgcCBpbiBzb3J0ZWQoaXRlcl9hdWRpbyhkYXRhX2RpcikpOgogICAgICAgIGgudXBkYXRlKHAucmVsYXRpdmVfdG8oZGF0YV9kaXIpLmFzX3Bvc2l4KCkuZW5jb2RlKCkpCiAgICAgICAgaC51cGRhdGUoc3RyKHAuc3RhdCgpLnN0X3NpemUpLmVuY29kZSgpKQogICAgcmV0dXJuIGguaGV4ZGlnZXN0KClbOjE2XQoKCmRlZiBhdWRpdChkYXRhX2Rpciwgc2FtcGxlX3NpemU9NDAwLCBmYXN0PUZhbHNlKToKICAgIGRhdGFfZGlyID0gUGF0aChkYXRhX2RpcikKICAgIGZpbGVzID0gbGlzdChpdGVyX2F1ZGlvKGRhdGFfZGlyKSkKICAgIGFzc2VydCBmaWxlcywgZiLZhNinINmF2YTZgdin2Kog2LXZiNiq2YrYqSDZgdmKIHtkYXRhX2Rpcn0iCgogICAgZm10cyA9IENvdW50ZXIocC5zdWZmaXgubG93ZXIoKSBmb3IgcCBpbiBmaWxlcykKICAgIHRvcF9kaXJzID0gQ291bnRlcihwLnBhcmVudC5yZWxhdGl2ZV90byhkYXRhX2RpcikuYXNfcG9zaXgoKSBpZiBwLnBhcmVudCAhPSBkYXRhX2RpciBlbHNlICIuIgogICAgICAgICAgICAgICAgICAgICAgIGZvciBwIGluIGZpbGVzKQoKICAgIHRvdGFsX2hvdXJzLCBkdXJzLCBiYWQgPSAwLjAsIFtdLCBbXQogICAgc2FtcGxlID0gZmlsZXMgaWYgZmFzdCBlbHNlIHJhbmRvbS5zYW1wbGUoZmlsZXMsIG1pbihzYW1wbGVfc2l6ZSwgbGVuKGZpbGVzKSkpCiAgICBmb3IgcCBpbiBzYW1wbGU6CiAgICAgICAgZCA9IGF1ZGlvX2R1cmF0aW9uKHApCiAgICAgICAgaWYgZCBpcyBOb25lOgogICAgICAgICAgICBiYWQuYXBwZW5kKHAubmFtZSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBkdXJzLmFwcGVuZChkKQogICAgaWYgZHVyczoKICAgICAgICBtZWFuX2R1ciA9IHN1bShkdXJzKSAvIGxlbihkdXJzKQogICAgICAgIGVzdF9ob3VycyA9IG1lYW5fZHVyICogbGVuKGZpbGVzKSAvIDM2MDAKICAgIGVsc2U6CiAgICAgICAgbWVhbl9kdXIsIGVzdF9ob3VycyA9IDAuMCwgMC4wCgogICAgaW5zaWRlID0gc3VtKDEgZm9yIGQgaW4gZHVycyBpZiBNSU5fRFVSIDw9IGQgPD0gTUFYX0RVUikKICAgIHJlcG9ydCA9IHsKICAgICAgICAibl9maWxlcyI6IGxlbihmaWxlcyksCiAgICAgICAgImZvcm1hdHMiOiBkaWN0KGZtdHMpLAogICAgICAgICJ0b3BfZGlycyI6IGRpY3QodG9wX2RpcnMubW9zdF9jb21tb24oMTApKSwKICAgICAgICAic2FtcGxlZCI6IGxlbihzYW1wbGUpLAogICAgICAgICJtZWFuX2R1cmF0aW9uIjogcm91bmQobWVhbl9kdXIsIDIpLAogICAgICAgICJlc3RfaG91cnMiOiByb3VuZChlc3RfaG91cnMsIDEpLAogICAgICAgICJwY3RfaW5fMV8zMHMiOiByb3VuZCgxMDAgKiBpbnNpZGUgLyBtYXgobGVuKGR1cnMpLCAxKSwgMSksCiAgICAgICAgInVucmVhZGFibGUiOiBiYWRbOjIwXSwKICAgICAgICAiZmluZ2VycHJpbnQiOiBmaW5nZXJwcmludChkYXRhX2RpciksCiAgICB9CiAgICBwcmludChqc29uLmR1bXBzKHJlcG9ydCwgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MikpCiAgICBwcmludChmIlxu2KPYrdi12YrZhtinIHtsZW4oZmlsZXMpfSDZhdmE2YHYp9mLINio2YXYqtmI2LPYtyDZhdiv2Kkge21lYW5fZHVyOi4xZn1zICIKICAgICAgICAgIGYiKNiq2YLYr9mK2LEge2VzdF9ob3VyczouMWZ9INiz2KfYudipKSIpCiAgICBwcmludChmIntyZXBvcnRbJ3BjdF9pbl8xXzMwcyddfSUg2K/Yp9iu2YQg2YbYt9in2YIgMeKAkzMwINir2KfZhtmK2KkgKNiu2KfYsdisINin2YTZhti32KfZgiDZitiq2KzYp9mH2YTZhyDYp9mE2YXYr9ix2KgpIikKICAgIGlmIGJhZDoKICAgICAgICBwcmludChmIuKaoCB7bGVuKGJhZCl9INmF2YTZgdin2Ysg2LrZitixINmC2KfYqNmEINmE2YTZgtix2KfYodipICjYudmK2YbYqSk6IHtiYWRbOjVdfSIpCiAgICByZXR1cm4gcmVwb3J0CgoKZGVmIG1ha2VfaG9sZG91dChkYXRhX2RpciwgY291bnQ9MjQsIHRlc3RfZGlyPU5vbmUsIHNlZWQ9NDIpOgogICAgIiIi2LnYstmEIGNvdW50INmF2YLYt9i52KfZiyDYudi02YjYp9im2YrYp9mLINil2YTZiSB0ZXN0X2RpciAo2YbZgtmE2Iwg2YTYpyDZhtiz2K4g4oCUINmF2YbYuSDYp9mE2KrYs9ix2YrYqCDYp9mE2YfZitmD2YTZiikuIiIiCiAgICBkYXRhX2RpciwgdGVzdF9kaXIgPSBQYXRoKGRhdGFfZGlyKSwgUGF0aCh0ZXN0X2RpciBvciAoUGF0aChkYXRhX2RpcikucGFyZW50IC8gImhvbGRvdXRfdGVzdCIpKQogICAgdGVzdF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgZXhpc3RpbmcgPSBsaXN0KGl0ZXJfYXVkaW8odGVzdF9kaXIpKQogICAgaWYgZXhpc3Rpbmc6CiAgICAgICAgcHJpbnQoZiJbaG9sZG91dF0g2YrZiNis2K8ge2xlbihleGlzdGluZyl9INmF2YLYt9i52KfZiyDZhdi52LLZiNmE2KfZiyDZhdiz2KjZgtin2Ysg2YHZiiB7dGVzdF9kaXJ9IOKAlCDZhNinINmG2YTZhdiz2YfYpyIpCiAgICAgICAgcHJpbnQoIltob2xkb3V0XSDZhNi52LLZhCDZhdis2YXZiNi52Kkg2KzYr9mK2K/YqTog2KfYrdiw2YEg2KfZhNmF2KzZhNivINij2Ygg2YXYsdixINmF2LPYp9ix2KfZiyDYrNiv2YrYr9in2YsiKQogICAgICAgIHJldHVybiBleGlzdGluZwoKICAgICMg2KfYrtiq2YrYp9ixINmF2YYg2YXZgtin2LfYuSDZhdiq2YjYs9i32Kkg2KfZhNi32YjZhCAoMuKAkzE1INir2KfZhtmK2KkpOiDZhdir2KfZhNmK2Kkg2YTZhNin2K7Yqtio2KfYsSDYp9mE2LPZhdi52YoKICAgIGNhbmRzID0gW10KICAgIGZpbGVzID0gbGlzdChpdGVyX2F1ZGlvKGRhdGFfZGlyKSkKICAgIHJhbmRvbS5zZWVkKHNlZWQpCiAgICByYW5kb20uc2h1ZmZsZShmaWxlcykKICAgIHVucmVhZGFibGUgPSAwCiAgICBmb3IgcCBpbiBmaWxlczoKICAgICAgICBpZiBsZW4oY2FuZHMpID49IGNvdW50OgogICAgICAgICAgICBicmVhawogICAgICAgIGQgPSBhdWRpb19kdXJhdGlvbihwKQogICAgICAgIGlmIGQgaXMgTm9uZToKICAgICAgICAgICAgdW5yZWFkYWJsZSArPSAxCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgMi4wIDw9IGQgPD0gMTUuMDoKICAgICAgICAgICAgY2FuZHMuYXBwZW5kKHApCiAgICAjINil2YYg2YTZhSDYqtmD2KrZgdmQINin2YTZhdmC2KfYt9i5INin2YTZhdiq2YjYs9i32KnYjCDYo9mD2YXZhCDZhdmGINij2Yog2LfZiNmEINi12KfZhNitCiAgICBpZiBsZW4oY2FuZHMpIDwgY291bnQ6CiAgICAgICAgZm9yIHAgaW4gZmlsZXM6CiAgICAgICAgICAgIGlmIGxlbihjYW5kcykgPj0gY291bnQ6CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICBpZiBwIGluIGNhbmRzOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgZCA9IGF1ZGlvX2R1cmF0aW9uKHApCiAgICAgICAgICAgIGlmIGQgaXMgTm9uZToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGlmIE1JTl9EVVIgPD0gZCA8PSBNQVhfRFVSOgogICAgICAgICAgICAgICAgY2FuZHMuYXBwZW5kKHApCgogICAgaWYgbm90IGNhbmRzOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgZiLZhNmFINmK2Y/Yudir2LEg2LnZhNmJINij2Yog2YXZgti32Lkg2LXYp9mE2K0g2YTZhNi52LLZhCAo2YHZj9it2LUge2xlbihmaWxlcyl92Iwg2LrZitixINmC2KfYqNmEINmE2YTZgtix2KfYodipIHt1bnJlYWRhYmxlfSkg4oCUICIKICAgICAgICAgICAgItin2YHYrdi1INin2YTYqNmK2KfZhtin2Kog2KjZgDogcHl0aG9uIGRhdGFfdXRpbHMucHkgYXVkaXQgIiArIHN0cihkYXRhX2RpcikpCiAgICBpZiB1bnJlYWRhYmxlOgogICAgICAgIHByaW50KGYiW2hvbGRvdXRdIOKaoCB7dW5yZWFkYWJsZX0g2YXZhNmB2KfZiyDYutmK2LEg2YLYp9io2YQg2YTZhNmC2LHYp9ih2Kkg2YTZhNmF2K/YqSDigJQg2KrYrNin2YfZjtmE2YfYpyDYp9mE2LnYstmEIikKCiAgICBtb3ZlZCA9IFtdCiAgICBmb3IgcCBpbiBjYW5kczoKICAgICAgICBkc3QgPSB0ZXN0X2RpciAvIHAubmFtZQogICAgICAgIHNodXRpbC5tb3ZlKHN0cihwKSwgc3RyKGRzdCkpCiAgICAgICAgbW92ZWQuYXBwZW5kKGRzdCkKICAgIHByaW50KGYiW2hvbGRvdXRdINi52Y/YstmEIHtsZW4obW92ZWQpfSDZhdmC2LfYudin2Ysg2KXZhNmJIHt0ZXN0X2Rpcn0iKQogICAgcHJpbnQoZiJbaG9sZG91dF0g2KjYtdmF2Kkg2KjZitin2YbYp9iqINin2YTYqtiv2LHZitioINio2LnYryDYp9mE2LnYstmEOiB7ZmluZ2VycHJpbnQoZGF0YV9kaXIpfSIpCiAgICB3aXRoIG9wZW4odGVzdF9kaXIgLyAiaG9sZG91dF9tYW5pZmVzdC5qc29uIiwgInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIGpzb24uZHVtcCh7Im1vdmVkX2Zyb20iOiBzdHIoZGF0YV9kaXIpLCAiZmlsZXMiOiBbbS5uYW1lIGZvciBtIGluIG1vdmVkXSwKICAgICAgICAgICAgICAgICAgICJzZWVkIjogc2VlZH0sIGYsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpCiAgICByZXR1cm4gbW92ZWQKCgpkZWYgdmVyaWZ5X2NrcHQocGF0aCk6CiAgICAiIiLZgdit2LUg2LPZhNin2YXYqSDZhtmC2LfYqSDYqtit2YLZgjog2KfZhNmF2YHYp9iq2YrYrdiMIGZpbml0ZdiMINit2KfZhNipINin2YTZhdit2LPZkdmG2Iwg2KfZhNiu2LfZiNipLiIiIgogICAgaW1wb3J0IHRvcmNoCiAgICBwYXRoID0gUGF0aChwYXRoKQogICAgYXNzZXJ0IHBhdGguZXhpc3RzKCksIGYi2KfZhNmF2YTZgSDYutmK2LEg2YXZiNis2YjYrzoge3BhdGh9IgogICAgc2l6ZV9nYiA9IHBhdGguc3RhdCgpLnN0X3NpemUgLyAxZTkKICAgIHN0YXRlID0gdG9yY2gubG9hZChwYXRoLCBtYXBfbG9jYXRpb249ImNwdSIpCiAgICBrZXlzID0gbGlzdChzdGF0ZS5rZXlzKCkpCiAgICBuZXRfa2V5cyA9IGxpc3Qoc3RhdGUuZ2V0KCJuZXQiLCB7fSkua2V5cygpKSBpZiAibmV0IiBpbiBzdGF0ZSBlbHNlIFtdCiAgICByZXBvcnQgPSB7ImZpbGUiOiBwYXRoLm5hbWUsICJzaXplX2diIjogcm91bmQoc2l6ZV9nYiwgMiksICJ0b3Bfa2V5cyI6IGtleXMsCiAgICAgICAgICAgICAgIm5ldF9tb2R1bGVzIjogbmV0X2tleXN9CiAgICBoYXNfb3B0ID0gIm9wdGltaXplciIgaW4gc3RhdGUgYW5kIGJvb2woc3RhdGUuZ2V0KCJvcHRpbWl6ZXIiKSkKICAgIHJlcG9ydFsiaGFzX29wdGltaXplcl9zdGF0ZSJdID0gaGFzX29wdAogICAgaWYgIml0ZXJzIiBpbiBzdGF0ZToKICAgICAgICByZXBvcnRbIml0ZXJzIl0gPSBzdGF0ZVsiaXRlcnMiXQogICAgaWYgImVwb2NoIiBpbiBzdGF0ZToKICAgICAgICByZXBvcnRbImVwb2NoIl0gPSBzdGF0ZVsiZXBvY2giXQoKICAgICMg2YHYrdi1IGZpbml0ZSDYudmE2Ykg2LnZitmG2Kkg2YXZhiDYp9mE2KPZiNiy2KfZhgogICAgbm9uX2Zpbml0ZSA9IDAKICAgIGNoZWNrZWQgPSAwCiAgICBmb3IgbWsgaW4gbmV0X2tleXM6CiAgICAgICAgc2QgPSBzdGF0ZVsibmV0Il1bbWtdCiAgICAgICAgZm9yIGksIChrLCB2KSBpbiBlbnVtZXJhdGUoc2QuaXRlbXMoKSk6CiAgICAgICAgICAgIGlmIG5vdCB0b3JjaC5pc190ZW5zb3Iodikgb3Igbm90IHYuaXNfZmxvYXRpbmdfcG9pbnQoKToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGNoZWNrZWQgKz0gMQogICAgICAgICAgICBpZiBub3QgdG9yY2guaXNmaW5pdGUodikuYWxsKCk6CiAgICAgICAgICAgICAgICBub25fZmluaXRlICs9IDEKICAgICAgICAgICAgaWYgY2hlY2tlZCA+PSAzMDA6CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgIGlmIGNoZWNrZWQgPj0gMzAwOgogICAgICAgICAgICBicmVhawogICAgcmVwb3J0WyJ3ZWlnaHRzX2NoZWNrZWQiXSA9IGNoZWNrZWQKICAgIHJlcG9ydFsibm9uX2Zpbml0ZV90ZW5zb3JzIl0gPSBub25fZmluaXRlCiAgICByZXBvcnRbInZlcmRpY3QiXSA9ICgi2LPZhNmK2YXYqSDinJMiIGlmIG5vbl9maW5pdGUgPT0gMCBlbHNlICLimqAg2KrZiNis2K8g2KPZiNiy2KfZhiDYutmK2LEg2LHZgtmF2YrYqSEiKQoKICAgIHByaW50KGpzb24uZHVtcHMocmVwb3J0LCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0yKSkKICAgIGlmIGhhc19vcHQ6CiAgICAgICAgcHJpbnQoIuKGkiDZgtin2KjZhNipINmE2YTYp9iz2KrYptmG2KfZgSDYp9mE2YPYp9mF2YQgKNij2YjYstin2YYgKyDZhdmP2K3Ys9mQ2ZHZhiArINmF2KzYr9mI2YQpIikKICAgIGVsc2U6CiAgICAgICAgcHJpbnQoIuKGkiDYo9mI2LLYp9mGINmB2YLYtyAo2YbZgti32Kkg2LHYs9mF2YrYqSDYo9mIIGZ0X21vZGVsKSDigJQg2KrYtdmE2K0g2YTYqNiv2KfZitipINis2K/Zitiv2Kkg2YTYpyDZhNin2LPYqtim2YbYp9mBINmD2KfZhdmEIikKICAgIHJldHVybiByZXBvcnQKCgpkZWYgbWFpbigpOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0i2KPYr9mI2KfYqiDYp9mE2KjZitin2YbYp9iqINmI2KfZhNmG2YLYp9i3IikKICAgIHN1YiA9IGFwLmFkZF9zdWJwYXJzZXJzKGRlc3Q9ImNtZCIsIHJlcXVpcmVkPVRydWUpCiAgICBhMSA9IHN1Yi5hZGRfcGFyc2VyKCJhdWRpdCIsIGhlbHA9Itiq2K/ZgtmK2YIg2YXYrNmE2K8g2KjZitin2YbYp9iqIikKICAgIGExLmFkZF9hcmd1bWVudCgiZGF0YV9kaXIiKQogICAgYTEuYWRkX2FyZ3VtZW50KCItLXNhbXBsZSIsIHR5cGU9aW50LCBkZWZhdWx0PTQwMCkKICAgIGExLmFkZF9hcmd1bWVudCgiLS1mYXN0IiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIGEyID0gc3ViLmFkZF9wYXJzZXIoImhvbGRvdXQiLCBoZWxwPSLYudiy2YQg2YXZgtin2LfYuSDYp9iu2KrYqNin2LEiKQogICAgYTIuYWRkX2FyZ3VtZW50KCJkYXRhX2RpciIpCiAgICBhMi5hZGRfYXJndW1lbnQoIi0tY291bnQiLCB0eXBlPWludCwgZGVmYXVsdD0yNCkKICAgIGEyLmFkZF9hcmd1bWVudCgiLS10ZXN0LWRpciIsIGRlZmF1bHQ9Tm9uZSkKICAgIGEyLmFkZF9hcmd1bWVudCgiLS1zZWVkIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NDIpCiAgICBhMyA9IHN1Yi5hZGRfcGFyc2VyKCJ2ZXJpZnkiLCBoZWxwPSLZgdit2LUg2YbZgti32Kkg2KrYrdmC2YIiKQogICAgYTMuYWRkX2FyZ3VtZW50KCJja3B0IikKICAgIGE0ID0gc3ViLmFkZF9wYXJzZXIoImZpbmdlcnByaW50IiwgaGVscD0i2KjYtdmF2Kkg2YXYrNmE2K8g2KjZitin2YbYp9iqIikKICAgIGE0LmFkZF9hcmd1bWVudCgiZGF0YV9kaXIiKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQoKICAgIGlmIGFyZ3MuY21kID09ICJhdWRpdCI6CiAgICAgICAgYXVkaXQoYXJncy5kYXRhX2RpciwgYXJncy5zYW1wbGUsIGFyZ3MuZmFzdCkKICAgIGVsaWYgYXJncy5jbWQgPT0gImhvbGRvdXQiOgogICAgICAgIG1ha2VfaG9sZG91dChhcmdzLmRhdGFfZGlyLCBhcmdzLmNvdW50LCBhcmdzLnRlc3RfZGlyLCBhcmdzLnNlZWQpCiAgICBlbGlmIGFyZ3MuY21kID09ICJ2ZXJpZnkiOgogICAgICAgIHZlcmlmeV9ja3B0KGFyZ3MuY2twdCkKICAgIGVsaWYgYXJncy5jbWQgPT0gImZpbmdlcnByaW50IjoKICAgICAgICBwcmludChmaW5nZXJwcmludChhcmdzLmRhdGFfZGlyKSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==",
}

written, skipped = [], []
for name, b64 in KIT_FILES.items():
    content = base64.b64decode(b64).decode("utf-8")
    dst = KIT_DIR / name
    new_hash = hashlib.sha256(content.encode()).hexdigest()[:12]
    if dst.exists():
        old = dst.read_text(encoding="utf-8")
        if hashlib.sha256(old.encode()).hexdigest()[:12] == new_hash:
            skipped.append(name)
            continue
    ast.parse(content)  # لا نكتب ملفاً مكسوراً أبداً
    dst.write_text(content, encoding="utf-8")
    written.append(name)

print(f"[kit] كُتب {len(written)}: {written}")
print(f"[kit] تُخطّي (مطابق): {len(skipped)}")
print(f"[kit] العدة في: {KIT_DIR}")
assert (KIT_DIR / "train_iraqi.py").exists(), "فشل تثبيت العدة!"
import sys; sys.path.insert(0, str(KIT_DIR))

<div dir="rtl">

## الخلية 5 — البيانات العراقية

مسارك الحالي يملك خط بيانات جاهزاً. هذه الخلية تحاول استخدام خط البيانات الموجود
في المستودع، وإن لم يوجد تستخدم مجلد بياناتك المخصص مباشرة.

**الحد الأدنى**: 1–30 ثانية للمقطع، صيغ wav/flac/mp3/m4a/opus/ogg، نظيفة بلا موسيقى.
</div>

In [ ]:
# ============================================================ #
#  الخلية 5 — جلب البيانات العراقية (استدعاء خط البيانات)      #
# ============================================================ #
import subprocess, sys, os
from pathlib import Path

AUD = {".wav", ".mp3", ".flac", ".ogg", ".opus", ".m4a"}

def count_audio(d):
    return sum(1 for p in Path(d).rglob("*") if p.suffix.lower() in AUD)

# مسار بديل: ضع بياناتك هنا (أو ارفعها إلى Drive واستخدم مسار Drive)
CUSTOM_DATA = os.environ.get("IRAQI_CUSTOM_DATA", "")

n_existing = count_audio(DATA_DIR) if DATA_DIR.exists() else 0
if n_existing >= 50:
    print(f"[data] البيانات موجودة: {n_existing} مقطعاً في {DATA_DIR} — لا شيء لفعله")
elif CUSTOM_DATA and Path(CUSTOM_DATA).exists():
    print(f"[data] استخدام مجلد مخصص: {CUSTOM_DATA}")
    DATA_DIR = Path(CUSTOM_DATA)
else:
    # محاولة استخدام خط البيانات المضمّن في مستودع رافدَين (إن نسخ إلى ROOT)
    PIPE = ROOT / "iraqi_pipeline.py"
    if PIPE.exists():
        print("[data] تشغيل خط البيانات (YODAS + المصادر النظيفة)...")
        sh([str(VENV_PY), str(PIPE), "--root", str(ROOT)], cwd=str(ROOT))
    else:
        print(f"""[data] لا بيانات بعد في {DATA_DIR}.
الخيارات:
  1) ارفع بياناتك إلى Drive ثم: os.environ['IRAQI_CUSTOM_DATA']='/content/drive/MyDrive/your_data' وأعد الخلية
  2) انسخ iraqi_pipeline.py إلى {ROOT} (خط بيانات رافدَين) وشغّل من جديد
  3) ارفع ملفات مباشرة إلى {DATA_DIR} عبر لوحة الملفات ثم أعد التشغيل""")
        raise SystemExit("لا بيانات — أكمل أحد الخيارات أعلاه ثم أعد الخلية")

n = count_audio(DATA_DIR)
print(f"[data] الإجمالي: {n} مقطعاً")
assert n >= 50, "أقل من 50 مقطعاً — بيانات غير كافية للتدريب المفيد"

<div dir="rtl">

## الخلية 6 — عزل الاختبار المعزول + تدقيق البيانات

**حماية نزاهة التقييم**: تُنقل (لا تُنسخ) 24 مقطعاً من التدريب إلى مجلد منفصل.
التقييم عليها فقط (خلية 9). إن سبق العزل، الخلية تكتشف وتتجاوز.
</div>

In [ ]:
# ============================================================ #
#  الخلية 6 — عزل الاختبار + تدقيق شامل                        #
# ============================================================ #
import sys, json
sys.path.insert(0, str(KIT_DIR))
from data_utils import audit, make_holdout, fingerprint

# 1) العزل (مرة واحدة — idempotent)
moved = make_holdout(DATA_DIR, count=int(CFG["HOLDOUT_COUNT"]), test_dir=str(TEST_DIR))
print()

# 2) التدقيق السريع (عينة 400)
report = audit(DATA_DIR, sample_size=400, fast=False)

# 3) حفظ بصمة البيانات على Drive (مرجع نزاهة دائم)
if DRIVE_RUN:
    (DRIVE_RUN / "data_fingerprint.json").write_text(
        json.dumps({"fingerprint": report["fingerprint"], "n_files": report["n_files"],
                    "est_hours": report["est_hours"], "ts": __import__("time").time()},
                   ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\n[holdout] بصمة البيانات محفوظة على Drive")

<div dir="rtl">

## الخلية 7 — التدريب 🚀 (اتركها تعمل)

الحارس يطلق المدرب المحصّن ويستأنفه تلقائياً حتى بلوغ `TOTAL_STEPS` التراكمي.
- **ميزانية الجلسة**: يتوقف نظيفاً قبل نفاد `TIME_BUDGET_H` ويحفظ نقطة أخيرة
- **مرآة Drive**: كل نقطة تُنسخ فوراً (آخر نقطتين)
- **إغلاق المتصفح آمن**: النواة تستمر (Colab يبقي الجلسة حية مع خلية نشطة)
- عند انقطاع الجلسة كلياً: أعد تشغيل الخلايا 1→7 بالترتيب — التدريب يستأنف من حيث توقف

**لا تُوقف هذه الخلية** إلا للتحقق من شيء عابر ثم أعد تشغيلها فوراً (الاستئناف آمن).
</div>

In [ ]:
# ============================================================ #
#  الخلية 7 — التدريب مع الحارس (استئناف كامل + ميزانية)      #
# ============================================================ #
import sys, os
sys.path.insert(0, str(KIT_DIR))
os.environ.setdefault("IRAQI_KIT_DIR", str(KIT_DIR))

from watchdog import run_training_forever, check_session_health

# نظرة على الوضع الحالي أولاً
check_session_health(drive_dir=str(DRIVE_RUN) if DRIVE_RUN else None)

CONFIG_YML = SEEDVC / "configs" / "presets" / "config_dit_mel_seed_uvit_whisper_base_f0_44k.yml"
assert CONFIG_YML.exists(), f"ملف الإعدادات غير موجود: {CONFIG_YML}"

run_training_forever(
    repo_dir=str(SEEDVC),
    config=str(CONFIG_YML),
    dataset_dir=str(DATA_DIR),
    run_name=str(CFG["RUN_NAME"]),
    total_steps=int(CFG["TOTAL_STEPS"]),
    batch_size=int(CFG["BATCH"]),
    save_every=int(CFG["SAVE_EVERY"]),
    num_workers=int(CFG["TRAIN_WORKERS"]),
    time_budget_hours=float(CFG["TIME_BUDGET_H"]),
    drive_dir=str(DRIVE_RUN) if DRIVE_RUN else None,
    keep_checkpoints=int(CFG["KEEP_CKPTS"]),
    kit_dir=str(KIT_DIR),
    validate_first=True,   # فحص السلامة قبل الانطلاق
)

<div dir="rtl">

## الخلية 8 — المراقبة الحية (شغّلها بجانب التدريب)

تعرض منحنى الخسارة والسرعة والإسقاط الزمني والحكم الصحي. لا تستهلك GPU.
</div>

In [ ]:
# ============================================================ #
#  الخلية 8 — المراقبة الحية                                   #
# ============================================================ #
import sys
sys.path.insert(0, str(KIT_DIR))
from loss_monitor import parse_history, parse_console, analyze, try_plot
from pathlib import Path

RUN_LOCAL = SEEDVC / "runs" / CFG["RUN_NAME"]

hist = RUN_LOCAL / "loss_history.jsonl"
console = RUN_LOCAL / "train_console.log"
if hist.exists():
    pts = parse_history(hist)
    sits, epoch_len = [], None
elif console.exists():
    text = console.read_text(encoding="utf-8", errors="replace")
    pts, sits, epoch_len = parse_console(text)
else:
    pts, sits, epoch_len = [], [], None
    print(f"لا سجل بعد في {RUN_LOCAL}")

if pts:
    r, report = analyze(pts, sits, epoch_len,
                        target=int(CFG["TOTAL_STEPS"]),
                        budget_hours=float(CFG["TIME_BUDGET_H"]))
    print(report)
    try_plot(pts, save_to=str(RUN_LOCAL / "loss_curve.png"))

<div dir="rtl">

## الخلية 9 — التقييم الموضوعي (جلسة مستقلة بعد التدريب)

**لا تشغّلها أثناء التدريب** — تشاركه GPU. شغّلها بعد اكتمال مرحلة أو انقطاع الجلسة:
تحوّل عينات الاختبار بكل نقطة (وخط الأساس الرسمي) عبر `inference.py` الرسمي،
ثم تقيس تشابه المتحدث (SIM) والطبيعية (UTMOS) وترتّب النقاط.

**قاعدة القرار**: أعلى SIM مع UTMOS لا ينهار (لا ينخفض > 0.2 عن خط الأساس).
</div>

In [ ]:
# ============================================================ #
#  الخلية 9 — التقييم الموضوعي (SIM + UTMOS + خط أساس)        #
# ============================================================ #
import sys, subprocess
sys.path.insert(0, str(KIT_DIR))

RUN_LOCAL = SEEDVC / "runs" / CFG["RUN_NAME"]
assert TEST_DIR.exists() and any(TEST_DIR.iterdir()), \
    "لا مقاطع اختبار معزولة — شغّل الخلية 6 أولاً"

cmd = [str(VENV_PY), str(KIT_DIR / "evaluate_ckpt.py"),
       "--run-dir", str(RUN_LOCAL),
       "--test-dir", str(TEST_DIR),
       "--max-ckpts", str(int(CFG["EVAL_MAX_CKPTS"])),
       "--n-sources", str(int(CFG["EVAL_SOURCES"])),
       "--n-refs", str(int(CFG["EVAL_REFS"])),
       "--diffusion-steps", str(int(CFG["DIFF_STEPS"]))]
if DRIVE_RUN:
    cmd += ["--drive-dir", str(DRIVE_RUN)]
# أضف نقاط gold إن وُجدت
for g in RUN_LOCAL.glob("gold_*.pth"):
    cmd += ["--extra-ckpt", str(g)]

print("$", " ".join(cmd))
r = subprocess.run(cmd, cwd=str(SEEDVC))
print(f"\n[eval] رمز الخروج: {r.returncode}")

<div dir="rtl">

## الخلية 10 — اعتماد النقطة الذهبية والنشر

بعد اختيار الفائز من التقييم (خلية 9): نسخ محمي إلى Drive باسم مميز
+ أمر الاستدلال الرسمي جاهزاً للنسخ.
</div>

In [ ]:
# ============================================================ #
#  الخلية 10 — اعتماد النقطة الذهبية                          #
# ============================================================ #
import shutil, json, subprocess, time
from pathlib import Path

RUN_LOCAL = SEEDVC / "runs" / CFG["RUN_NAME"]

# عدّل هذا بعد قراءة نتائج الخلية 9 — مثال: "DiT_epoch_00000_step_006000"
GOLD_STEP = None   # None = آخر نقطة موجودة

ckpts = sorted(RUN_LOCAL.glob("DiT_epoch_*_step_*.pth"),
               key=lambda p: int(re.search(r"step_(\d+)", p.stem).group(1)))
assert ckpts, "لا نقاط"
src = next((c for c in ckpts if GOLD_STEP and str(GOLD_STEP) in c.name), ckpts[-1])

stamp = time.strftime("%Y%m%d")
gold_name = f"gold_{CFG['RUN_NAME']}_s{re.search(r'step_(\d+)', src.stem).group(1)}_{stamp}.pth"
gold_local = RUN_LOCAL / gold_name
shutil.copy2(src, gold_local)
print(f"[gold] النقطة المعتمدة: {src.name} → {gold_name}")

if DRIVE_RUN:
    shutil.copy2(gold_local, DRIVE_RUN / gold_name)
    # حماية: نسخة الإعدادات والبصمة معها دائماً
    for aux in RUN_LOCAL.glob("config_*.yml"):
        shutil.copy2(aux, DRIVE_RUN / aux.name)
    print(f"[gold] ✓ محفوظة على Drive: {DRIVE_RUN / gold_name}")

# أمر الاستدلال الرسمي جاهز للنسخ
cfg_yml = next(RUN_LOCAL.glob("config_*.yml"), CONFIG_YML)
print(f"""
# ===== أمر الاستدلال الرسمي (انسخه) =====
cd {SEEDVC}
{VENV_PY} inference.py \\
  --source مقطع_المصدر.wav --target صوت_المرجع.wav \\
  --output ./out --config {cfg_yml} \\
  --checkpoint {gold_local} \\
  --diffusion-steps 30 --f0-condition True --auto-f0-adjust True --fp16 True
""")

<div dir="rtl">

## الخلية 11 — استكشاف الأخطاء السريع

| العَرَض | السبب الأرجح | الحل |
|---|---|---|
| خطأ CUDA OOM | دفعة أو مقاطع أطول من ذاكرة T4 | أعد التشغيل؛ إن تكرر: `CFG['BATCH']=1` |
| توقف عند تنزيل نموذج HF | شبكة/قيود HF | أعد الخلية 3 (الكاش محفوظ)، أو `HF_ENDPOINT=https://hf-mirror.com` |
| "لا نقاط على Drive" | Drive غير موصول أو مسار خاطئ | شغّل الخلية 2 وتحقق من `DRIVE_OK=True` |
| الخسارة NaN | بيانات فيها مقاطع فاسدة | الحارس تخطّاها تلقائياً؛ إن تكرر 3×: راجع `data_utils.audit` |
| الجلسة انقطعت | حد Colab المجاني | أعد تشغيل الخلايا 1→7 بالترتيب — الاستئناف تلقائي |
| فقدت نقاط بعد استئناف | (أُصلح في هذه العدة) | تأكد أن التدريب يمر عبر `train_iraqi.py` وليس train.py الرسمي |

**تحقق أخير**: بصمة البيانات يجب أن تتطابق مع المحفوظة على Drive —
أي اختلاف يعني تغيّرت بيانات التدريب (وعليك إعادة التقييم).
</div>

In [ ]:
# ============================================================ #
#  الخلية 11 — فحوص سريعة عند الشك                            #
# ============================================================ #
import sys, json, subprocess
sys.path.insert(0, str(KIT_DIR))
from pathlib import Path
from data_utils import verify_ckpt, fingerprint

RUN_LOCAL = SEEDVC / "runs" / CFG["RUN_NAME"]

print("=== 1) أحدث نقطة: فحص سلامة كامل ===")
ckpts = sorted(RUN_LOCAL.glob("DiT_epoch_*_step_*.pth"),
               key=lambda p: int(re.search(r"step_(\d+)", p.stem).group(1)))
if ckpts:
    verify_ckpt(ckpts[-1])
else:
    print("لا نقاط بعد")

print("\n=== 2) بصمة البيانات الحالية ===")
fp_now = fingerprint(DATA_DIR)
print(f"الحالية: {fp_now}")
fp_saved = DRIVE_RUN / "data_fingerprint.json" if DRIVE_RUN else None
if fp_saved and fp_saved.exists():
    fp_old = json.loads(fp_saved.read_text(encoding="utf-8"))["fingerprint"]
    print(f"المحفوظة: {fp_old}")
    print("⚠ تغيّرت بيانات التدريب! أعد التقييم من الصفر" if fp_old != fp_now
          else "✓ متطابقة")

print("\n=== 3) حالة Drive ===")
if DRIVE_RUN:
    for p in sorted(DRIVE_RUN.glob("*.pth")):
        print(f"  {p.name}  ({p.stat().st_size/1e9:.2f}GB)")
    for aux in DRIVE_RUN.glob("*.json"):
        print(f"  {aux.name}")

print("\n=== 4) زمن بقاء الجلسة (تقديري من نبضة القلب) ===")
hb = RUN_LOCAL / "heartbeat.json"
if hb.exists():
    h = json.loads(hb.read_text())
    age_min = int((__import__("time").time() - h["ts"]) / 60)
    print(f"آخر حفظ: خطوة {h['step']} قبل {age_min} دقيقة")
    budget_min = CFG["TIME_BUDGET_H"] * 60
    print(f"ميزانية الجلسة: {budget_min:.0f} دقيقة — "
          + ("✓ الوقت متاح" if age_min < budget_min * 0.8 else "⚠ قاربت النفاد — الحارس سيحفظ ويتوقف نظيفاً"))